# Matriz experimental SIBGRAPI — Puzzle Mix × Decomposição × Explicabilidade

Adaptado de `pipeline_resnet18.ipynb` (ablação de schedulers).
**Resumo completo do que mudou** está no docstring da primeira célula de
código abaixo (removido / adicionado / mantido, com o motivo de cada
mudança). Testado de ponta a ponta (baseline, cutmix, puzzlemix, resume e
resumo estatístico) com um dataset sintético antes da entrega.

Pontos-chave:

- Ablação de scheduler **removida** (decisão anterior, sem diferença
  significativa entre eles) → scheduler fixo em CosineAnnealingLR.
- **Arquiteturas** generalizadas: `build_model(arch_name, ...)` no lugar do
  antigo `build_model()` fixo em ResNet18. Vem com **VGG16-BN** habilitada;
  adicionar a segunda arquitetura é só incluir o branch em `build_model()`
  e o nome em `ARCHITECTURES`.
- **Regimes de treino** novos: `baseline`, `cutmix`, `puzzlemix` — o eixo
  central do estudo.
- Cabeça do VGG16 trocada por *global average pooling* + uma FC (em vez das
  duas FC de 4096 originais, ~120M parâmetros) — sem isso, só o otimizador
  (AdamW, 2 estados/parâmetro) já não cabe em 6GB de VRAM. Achado confirmado
  no teste: a versão original travava por memória mesmo num smoke test
  minúsculo.

In [ ]:
# Setup de caminhos: acha a raiz do repositorio, roda a partir dela
# (Originais/, PuzzleMix/ e sibgrapi_ckpt_*/ ficam na raiz) e importa de src/.
import os, sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "regimes.py").is_file())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

In [ ]:
"""
=============================================================================
MATRIZ EXPERIMENTAL SIBGRAPI - Puzzle Mix x Decomposicao de Tensores x
Explicabilidade / Displasia Epitelial Oral (4 classes)
=============================================================================

Adaptado de: pipeline_resnet18.ipynb (ablacao de schedulers).

O QUE MUDOU EM RELACAO AO NOTEBOOK ORIGINAL
--------------------------------------------------------------------------
REMOVIDO:
    - Ablacao de SCHEDULERS (CosineAnnealingLR vs OneCycleLR vs
      ReduceLROnPlateau). Decisao anterior (ablacao multi-seed, Friedman
      p=0.623): nao ha diferenca estatisticamente significativa entre eles.
      O scheduler fica FIXO em CosineAnnealingLR daqui em diante, para nao
      gastar orcamento de computo respondendo uma pergunta ja respondida.
    - build_model() especifico de ResNet18.

ADICIONADO:
    - Suporte a MULTIPLAS ARQUITETURAS (ARCHITECTURES): VGG16-BN e,
      quando definida, a segunda arquitetura de comparacao. Escolhidas por
      operarem no MESMO tipo de camada-alvo da decomposicao (convs 3x3
      densas) - ConvNeXt/EfficientNet/MobileNet foram descartadas por usarem
      depthwise conv, que nao e' comparavel ao Tucker/TT usado no VGG.
    - REGIMES DE TREINO (TRAINING_REGIMES): "baseline", "cutmix",
      "puzzlemix". CutMix entra como braco de controle (mistura espacial
      SEM saliencia), para isolar se o ganho do Puzzle Mix vem da saliencia
      ou so' do fato de misturar imagens.
    - Implementacao de CutMix (padrao, Yun et al. 2019) e de uma versao
      SIMPLIFICADA de Puzzle Mix (Kim et al., ICML 2020) guiada por
      saliencia de gradiente de entrada + corte minimo (alpha-beta swap,
      ver "CORRIGIDO" abaixo). Ver docstring da secao 3 para as
      simplificacoes assumidas frente ao metodo original.
    - Deteccao automatica, por arquitetura, da camada-alvo do Grad-CAM
      (GRADCAM_LAYER_BY_ARCH) e da lista de convs decomponiveis (3x3,
      excluindo 1x1/transicoes) - metadados salvos no CSV para os notebooks
      seguintes (decomposicao + explicabilidade) nao precisarem redescobrir
      isso.
    - LR_BY_ARCH_REGIME: placeholder para os resultados do HPO curto (fase
      0b, Optuna, 10 trials por braco) - o LR=1e-3 herdado do ResNet18 NAO
      e' assumido valido para VGG/DenseNet (ver ressalva anterior sobre
      faixas de LR). Preencha este dict apos rodar o HPO; ate' la', o
      pipeline avisa e usa o valor default (mais conservador) por arquitetura.
    - BATCH_SIZE_BY_ARCH e ACCUM_STEPS_BY_ARCH: VGG16 tem ativações
      grandes nas primeiras camadas (mais VRAM por imagem que ResNet);
      Puzzle Mix soma um forward+backward extra (saliencia) por batch. Em
      6GB isso pode nao caber com batch 32 -> reduz batch fisico e usa
      accumulation para manter o batch EFETIVO comparavel entre arquiteturas.
    - USE_AMP (mixed precision) ligado por padrao, mesma razao acima.
    - MIX_PROB e MIXUP_ALPHA: fracao de batches efetivamente misturados e
      parametro da Beta(alpha, alpha) que sorteia lambda (convencao usual
      de mixup/cutmix/puzzlemix).
    - RESTRICT_ADJACENT_MIX: flag para a decisao pendente sobre misturar
      apenas classes ordinais vizinhas (|rank_a - rank_b| <= 1) ou permitir
      qualquer par. Default False (permite qualquer par); decidir na fase 0b.
    - FIELDNAMES do CSV: "Scheduler" (e as colunas de ablacao correspondentes)
      trocado por "Architecture" + "Training_Regime"; get_completed_configs()
      agora resume por (Architecture, Training_Regime, Seed).
    - SEEDS_BY_ARCH: permite menos seeds na arquitetura secundaria (economia
      de orcamento combinatorio), conforme discutido.

CORRIGIDO (checagem de 2026-08-12/13, ver puzzlemix_sanity_check/RELATORIO.md
no repo para o diagnostico completo com imagens):
    - input_saliency(): usava soma de |dL/dx| nos canais (norma L1 por
      canal); trocado para ||dL/dx||_2 real (sqrt da soma dos quadrados),
      para bater com a notacao da metodologia. Muda ~2-6% dos blocos
      escolhidos pela mascara nas imagens testadas - nao e' so' cosmetico.
    - BUG CONFIRMADO em _ot_block_mask (removida): o plano de transporte
      otimo (POT) era calculado corretamente, mas a decisao "fica com A"
      usava diag(plano), que e' degenerada (transportar um bloco pra ele
      mesmo custa zero, entao o LP marca ~100% dos blocos como "fica com A"
      quase sempre, para qualquer par de imagens/lambda testado). Resultado:
      a mascara final do braco "com transporte otimo" saia IDENTICA ao
      fallback guloso em 20/20 configuracoes testadas (4 pares x 5 lambdas)
      - PUZZLEMIX_USE_OT=True nao mudava nada na pratica.
    - ADICIONADO um passo de suavidade espacial de verdade
      (_alpha_beta_swap_mask, secao 3): a mascara por bloco agora vem de um
      corte minimo (grafo grade 4-conectada + arestas extras nos pares que o
      plano de OT realmente conectou), em vez de um ranking guloso por bloco
      isolado - e' a regularizacao "mask-graph contra mascara fragmentada"
      que a versao anterior deste notebook ja citava na docstring mas nao
      implementava. Resolvido com PyMaxflow (nao pyGCO): pyGCO/gco-wrapper
      nao tem wheel pre-compilada pra Windows e exige MSVC Build Tools pra
      compilar do fonte (falhou neste ambiente - ver timing.py). Para K=2
      rotulos (bloco vem de A ou de B) isso e' matematicamente equivalente
      ao que pyGCO calcularia: o algoritmo geral de troca alpha-beta so'
      itera entre PARES de rotulos: com apenas 2 rotulos ha' um unico par,
      entao um corte minimo binario ja' e' a solucao exata.
    - IMPORTANTE: as seeds de 'puzzlemix' ja' logadas em sibgrapi_logs.csv
      (antes desta correcao) usaram o fallback guloso disfarcado de "com
      OT" (ver bug acima) e SEM a suavidade espacial nova - precisam ser
      re-rodadas antes de qualquer conclusao publicavel sobre o regime
      puzzlemix.

MANTIDO (sem mudanca de logica):
    - Split holdout estratificado + 5-fold CV + ensemble no teste.
    - Metricas clinicas (QWK, danger_rate, underest_rate) e
      CHECKPOINT_METRIC="clinical_composite".
    - Mapeamento ordinal explicito (ORDINAL_ORDER) para QWK/erros perigosos.
    - Pipeline de aumento de dados original (Resize/Crop/flips/rotacao/
      ColorJitter) permanece identico e roda ANTES da mistura (CutMix/
      Puzzle Mix entram sobre o batch ja' transformado, dentro do loop de
      treino - nunca no conjunto de teste).
    - Telemetria por fold, resume via CSV, correcao da duplicacao de
      avaliacao de teste, num_workers=0 (Windows/Jupyter).

Uso:
    Rodar sequencialmente. main() itera
    ARCHITECTURES x TRAINING_REGIMES x SEEDS_BY_ARCH[arch] e faz append no
    mesmo CSV, pulando combinacoes ja concluidas (resume).

    Decomposicao de tensores (Tucker/TT) e as metricas de explicabilidade
    (Grad-CAM, estabilidade, deletion/insertion) ficam em notebooks
    separados, como no projeto anterior (pipeline_decomposition.ipynb) -
    este notebook so' produz os checkpoints e os metadados (camada-alvo,
    lista de convs decomponiveis) que esses notebooks vao consumir.
=============================================================================
"""

import os
import csv
import random
import warnings
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    confusion_matrix, f1_score, roc_auc_score, accuracy_score, cohen_kappa_score
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, models, transforms

try:
    import ot  # POT: Python Optimal Transport (usado no Puzzle Mix)
    POT_AVAILABLE = True
except ImportError:
    POT_AVAILABLE = False
    warnings.warn(
        "Pacote 'POT' (pip install POT) nao encontrado; Puzzle Mix vai usar "
        "o corte minimo so' com suavidade de grade (sem arestas informadas "
        "por transporte otimo). Instale POT para a versao mais fiel ao "
        "metodo original."
    )

try:
    import maxflow  # PyMaxflow: corte minimo / max-flow (usado no Puzzle Mix)
    MAXFLOW_AVAILABLE = True
except ImportError:
    MAXFLOW_AVAILABLE = False
    warnings.warn(
        "Pacote 'PyMaxflow' (pip install PyMaxflow) nao encontrado; Puzzle "
        "Mix vai usar o fallback guloso (greedy, sem suavidade espacial) em "
        "vez do corte minimo (alpha-beta swap). Instale PyMaxflow para a "
        "mascara com regularizacao 'mask-graph' contra fragmentacao."
    )

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
PLOTTING_AVAILABLE = True

In [ ]:
# =============================================================================
# 1. CONFIGURACAO DA MATRIZ EXPERIMENTAL
# =============================================================================
# --- Arquiteturas (NOVO: substitui a ablacao de scheduler) ---
# Ambas operam sobre convs 3x3 densas (mesma "intervencao" de decomposicao).
# Adicione "densenet121" (ou outra) aqui quando a segunda arquitetura for
# definida; get_decomposable_layers()/get_gradcam_layer() ja' tratam DenseNet
# de forma generica (ver secao 2).
ARCHITECTURES = ["vgg16_bn"]

# --- Regimes de treino (NOVO: o eixo central do estudo) ---
TRAINING_REGIMES = ["baseline", "cutmix", "puzzlemix"]

# --- Scheduler FIXO (REMOVIDO: nao e' mais ablacionado, ver docstring) ---
SCHEDULER_NAME = "CosineAnnealingLR"

# --- Seeds: 5 na arquitetura principal, 3 na(s) secundaria(s) (NOVO) ---
SEEDS_PRIMARY   = [42, 17, 7, 12, 123]
SEEDS_SECONDARY = [42, 17, 123]
SEEDS_BY_ARCH = {
    "vgg16_bn": SEEDS_PRIMARY,
    # arquiteturas adicionadas em ARCHITECTURES caem aqui por default:
}
DEFAULT_SEEDS_FOR_NEW_ARCH = SEEDS_SECONDARY

# --- Hiperparametros base ---
# ATENCAO: 1e-3 era calibrado para ResNet18 (ablacao de scheduler). NAO
# assumir valido para VGG/DenseNet - preencher LR_BY_ARCH_REGIME apos a
# fase 0b (HPO curto, Optuna, 10 trials/braco, fold 1/seed 42). Ate' la',
# o pipeline cai no default por arquitetura abaixo e avisa no log.
LR_DEFAULT_BY_ARCH = {
    "vgg16_bn": 5e-4,      # ponto medio da faixa sugerida (1e-4 a 1e-2)
    "densenet121": 5e-4,   # ponto medio da faixa sugerida (1e-4 a 5e-3)
}
LR_BY_ARCH_REGIME = {
    # Preencher apos o HPO curto, ex.: ("vgg16_bn", "puzzlemix"): 3.2e-4
}
WEIGHT_DECAY = 1e-5
EPOCHS       = 50
N_FOLDS      = 5
TEST_SIZE    = 0.2

# Checkpoint: mantido igual ao pipeline anterior.
CHECKPOINT_METRIC = "clinical_composite"  # opcoes: "auc", "f1", "qwk", "clinical_composite"
DANGER_RATE_WEIGHT = 1.0
UNDEREST_RATE_WEIGHT = 1.0
SAVE_TELEMETRY = True

# --- Batch size / memoria (NOVO: VGG16 + Puzzle Mix e' o caso mais apertado) ---
# BATCH_SIZE_BY_ARCH = tamanho FISICO do batch (o que efetivamente vai pra GPU).
# ACCUM_STEPS_BY_ARCH = passos de acumulacao de gradiente para manter o
# batch EFETIVO em 32 (== BATCH_SIZE_BY_ARCH * ACCUM_STEPS_BY_ARCH), o que
# preserva a comparabilidade entre arquiteturas apesar do batch fisico menor.
BATCH_SIZE_BY_ARCH = {
    "vgg16_bn": 16,
    "densenet121": 32,
}
ACCUM_STEPS_BY_ARCH = {
    "vgg16_bn": 2,
    "densenet121": 1,
}
EFFECTIVE_BATCH_SIZE = 32  # esperado para todas as arquiteturas; ver assert em run_config()
USE_AMP = True  # mixed precision - reduz VRAM, mesma razao acima

# --- Mistura (CutMix / Puzzle Mix) (NOVO) ---
MIXUP_ALPHA = 1.0   # parametro da Beta(alpha, alpha) para sortear lambda
MIX_PROB = 0.5      # fracao de batches efetivamente misturados (o resto treina "baseline" naquele batch)
PUZZLEMIX_GRID = 14  # grid de blocos para a mascara de saliencia (224/14 = blocos de 16x16 px)

# --- Puzzle Mix: corte minimo (alpha-beta swap) + transporte otimo (CORRIGIDO 2026-08-13) ---
# A mascara por bloco vem de um corte minimo (grafo grade 4-conectada, ver
# secao 3 / _alpha_beta_swap_mask). PUZZLEMIX_USE_OT so' controla se o plano
# de transporte otimo (POT) entra como arestas EXTRAS do grafo (estrutura
# espacial real do transporte) - ele nunca decide sozinho o rotulo de um
# bloco (isso era o bug da versao anterior, _ot_block_mask). Sem POT
# instalado, ou com PUZZLEMIX_USE_OT=False, o corte minimo roda so' com a
# suavidade de grade. Sem PyMaxflow instalado (MAXFLOW_AVAILABLE=False),
# cai no fallback guloso antigo (sem suavidade nenhuma).
PUZZLEMIX_USE_OT = True
# Peso da suavidade espacial (grade 4-conectada) no corte minimo, como
# fracao do desvio-padrao do 'gain' (sal_a - sal_b) por amostra - precisa
# ser relativo, nao um valor absoluto fixo, porque a escala do gain depende
# da formula de saliencia/normalizacao (um peso fixo grande satura a
# mascara inteira num unico rotulo; testado e descartado, ver
# puzzlemix_sanity_check/). 0.0 desliga a suavidade (~equivalente ao
# guloso, so' que ainda resolvido via corte minimo).
PUZZLEMIX_SMOOTH_FRAC = 0.35
# Peso extra (mesma escala relativa) nas arestas do grafo entre blocos que
# o transporte otimo de fato conectou (plano esparso: no maximo n+m-1
# entradas nao-nulas). So' tem efeito com PUZZLEMIX_USE_OT=True e POT
# instalado.
PUZZLEMIX_OT_EDGE_FRAC = 0.7

# Decisao PENDENTE (fase 0b): restringir mistura a classes ordinais vizinhas
# (|rank_a - rank_b| <= 1) para evitar rotulos intermediarios sem
# correspondencia clinica real. Default False = permite qualquer par.
# Rode os dois valores na fase 0b antes de fixar qual entra no artigo.
RESTRICT_ADJACENT_MIX = False

# DataLoader workers: 0 = roda no processo principal (ver nota do notebook
# original sobre Windows/Jupyter + num_workers>0).
NUM_WORKERS = 0

BASE_DIR   = "Originais"   # ImageFolder, 4 classes, sem split previo
OUTPUT_DIR = f"./sibgrapi_ckpt_{CHECKPOINT_METRIC}"
CSV_FILE   = os.path.join(OUTPUT_DIR, "sibgrapi_logs.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ordem clinica das classes (rank ordinal). Usada p/ QWK, erros perigosos e
# para restringir pares de mistura a classes vizinhas (RESTRICT_ADJACENT_MIX).
ORDINAL_ORDER = ["healthy", "mild", "moderate", "severe"]

# Determinismo: mesma logica do notebook original.
DETERMINISTIC = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando em: {device}")

In [ ]:
# =============================================================================
# 2. UTILITARIOS
# =============================================================================
def seed_everything(seed: int):
    """Fixa todas as fontes de aleatoriedade para reprodutibilidade entre runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

In [ ]:
def make_loader_generator(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# =============================================================================
# ARQUITETURAS (NOVO: generaliza build_model(), antes hardcoded p/ ResNet18)
# =============================================================================
def build_model(arch_name: str, num_classes: int):
    if arch_name == "vgg16_bn":
        model = models.vgg16_bn(weights=models.VGG16_BN_Weights.DEFAULT)
        # A "classifier" padrao do VGG (FC 25088->4096->4096->1000) tem
        # ~120M parametros - sozinha, mais pesada que a ResNet50 inteira.
        # Com AdamW (2 estados por parametro) isso sozinho consome ~2GB de
        # memoria de otimizador, o que estoura 6GB de VRAM somado as
        # ativacoes e ao passo extra de saliencia do Puzzle Mix. Trocamos
        # por global average pooling + uma unica FC, como fizemos no artigo
        # do SEMAT/decomposicao: cai para ~2 mil parametros na cabeca, sem
        # mudar as convs (que sao o alvo real da decomposicao/explicabilidade).
        model.avgpool = nn.AdaptiveAvgPool2d(1)
        model.classifier = nn.Linear(512, num_classes)
        return model

    if arch_name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(
        f"Arquitetura desconhecida: {arch_name}. "
        f"Adicione o branch de construcao em build_model() antes de usa-la "
        f"em ARCHITECTURES."
    )


def make_scheduler(optimizer):
    """
    Scheduler FIXO em CosineAnnealingLR (ver docstring: a ablacao anterior
    nao encontrou diferenca significativa entre schedulers). Mantido como
    funcao (em vez de instanciar direto em run_config) so' para deixar
    explicito, em um unico lugar, qual e' a escolha e por que.
    """
    return CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-8)


def ordinal_ranks(class_names):
    """Mapeia o indice de classe do ImageFolder -> rank ordinal clinico (0..3)."""
    missing = [c for c in class_names if c not in ORDINAL_ORDER]
    if missing:
        raise ValueError(
            f"Classes sem rank ordinal: {missing}. Ajuste ORDINAL_ORDER={ORDINAL_ORDER}."
        )
    return np.array([ORDINAL_ORDER.index(c) for c in class_names])


# =============================================================================
# METADADOS POR ARQUITETURA (NOVO)
# Usados pelos notebooks seguintes (decomposicao + explicabilidade), que nao
# devem precisar redescobrir isso: salvos no CSV a cada linha.
# =============================================================================
def get_decomposable_layers(model, arch_name):
    """
    Lista os nomes das convs 3x3 densas (kernel_size == (3,3)) elegiveis
    para Tucker/Tensor-Train, EXCLUINDO 1x1 (bottlenecks/transicoes) -
    decisao de comparabilidade: mesma intervencao nas duas arquiteturas.
    """
    names = []
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d) and mod.kernel_size == (3, 3):
            names.append(name)
    if not names:
        warnings.warn(f"Nenhuma conv 3x3 encontrada em '{arch_name}' - confira a arquitetura.")
    return names


def get_gradcam_layer(arch_name):
    """
    Retorna o NOME (string, resolvivel via model.get_submodule) da camada-
    alvo do Grad-CAM para cada arquitetura. Escolhida como a ativacao
    espacial de mais alta resolucao logo antes do pooling global, para
    manter os mapas comparaveis entre arquiteturas.

    VGG16-BN:  features[42] = ReLU imediatamente apos a ultima conv (BN+ReLU
               da ultima conv 3x3, antes do MaxPool final). Indice conferido
               manualmente contra a Sequential do torchvision - se a versao
               do torchvision mudar a composicao de vgg16_bn, reconferir.
    DenseNet121: "features" (saida do denseblock4 + norm5, ANTES do relu
               final que o forward() do DenseNet aplica manualmente - quem
               for extrair Grad-CAM aqui precisa aplicar F.relu no output
               capturado pelo hook, replicando torchvision.models.densenet).
    """
    if arch_name == "vgg16_bn":
        return "features.42"
    if arch_name == "densenet121":
        return "features"
    raise ValueError(
        f"Sem camada de Grad-CAM definida para '{arch_name}'. "
        f"Adicione o caso em get_gradcam_layer()."
    )

In [ ]:
def clinical_metrics(y_true, y_pred, rank_lookup):
    """
    QWK (quadratic), acuracia, erros perigosos e subestimacao clinica.
    Erro perigoso  = distancia ordinal >= 2 (em qualquer direcao).
    Subestimacao   = prever classe MENOS grave que a real, com distancia >= 2
                     (clinicamente o pior caso: chamar de saudavel um doente).
    """
    yt = rank_lookup[np.asarray(y_true)]
    yp = rank_lookup[np.asarray(y_pred)]
    n = len(yt)

    qwk = cohen_kappa_score(yt, yp, weights="quadratic")
    acc = accuracy_score(yt, yp)

    dist = np.abs(yt - yp)
    danger = int((dist >= 2).sum())
    under = int(((yp < yt) & (dist >= 2)).sum())
    return {
        "qwk": qwk,
        "acc": acc,
        "danger_count": danger,
        "danger_rate": danger / n,
        "underest_count": under,
        "underest_rate": under / n,
    }


def checkpoint_selection_score(val_auc, val_f1, y_true, y_pred, rank_lookup):
    """Score usado para escolher o checkpoint salvo em cada fold."""
    clin = clinical_metrics(y_true, y_pred, rank_lookup)
    metric = CHECKPOINT_METRIC.lower()
    if metric == "auc":
        score = val_auc if not np.isnan(val_auc) else -np.inf
    elif metric == "f1":
        score = val_f1
    elif metric == "qwk":
        score = clin["qwk"]
    elif metric == "clinical_composite":
        score = (clin["qwk"]
                 - DANGER_RATE_WEIGHT * clin["danger_rate"]
                 - UNDEREST_RATE_WEIGHT * clin["underest_rate"])
    else:
        raise ValueError(
            "CHECKPOINT_METRIC deve ser: 'auc', 'f1', 'qwk' ou 'clinical_composite'."
        )
    if np.isnan(score):
        score = -np.inf
    return score, clin

In [ ]:
def plot_fold_telemetry(history: dict, fold: int, arch_name: str, regime: str, seed: int,
                        save_dir: str = OUTPUT_DIR):
    """
    Gera um painel com 3 graficos de linha para auditar o treinamento do fold.
    history deve conter: 'train_loss', 'val_loss', 'train_acc', 'val_acc', 'lrs'

    MUDANCA vs. original: titulo/nome de arquivo passam a identificar
    (arquitetura, regime de treino, seed) em vez de (scheduler, seed), ja
    que o scheduler agora e' fixo (CosineAnnealingLR) para todas as runs.
    """
    if not SAVE_TELEMETRY:
        return None
    if not PLOTTING_AVAILABLE:
        warnings.warn("matplotlib indisponivel; telemetria visual nao sera salva.")
        return None

    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=120)
    fig.suptitle(f"Auditoria de Treinamento - {arch_name} | {regime} | Seed {seed} | Fold {fold}",
                 fontsize=16, fontweight='bold')

    # --- Painel 1: Loss ---
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], label='Validation Loss', color='#ff7f0e', linewidth=2)
    axes[0].set_title("Evolucao da Funcao de Custo (CrossEntropy, possivelmente misturada)")
    axes[0].set_xlabel("Epocas")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend()

    # --- Painel 2: Acuracia ---
    # NOTA: em batches misturados (CutMix/Puzzle Mix), train_acc e' calculada
    # contra o rotulo PRIMARIO (y_a) do par - e' apenas um sinal diagnostico
    # de treino, nao uma metrica de avaliacao (val/test nunca sao misturados).
    axes[1].plot(epochs, history['train_acc'], label='Train Accuracy (vs. y_a)', color='#1f77b4', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], label='Validation Accuracy', color='#2ca02c', linewidth=2)
    axes[1].set_title("Evolucao da Acuracia")
    axes[1].set_xlabel("Epocas")
    axes[1].set_ylabel("Acuracia")
    axes[1].set_ylim([0.0, 1.05])
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()

    # --- Painel 3: Learning Rate ---
    axes[2].plot(epochs, history['lrs'], color='#9467bd', linewidth=2)
    axes[2].set_title(f"Evolucao do Learning Rate ({SCHEDULER_NAME}, fixo)")
    axes[2].set_xlabel("Epocas")
    axes[2].set_ylabel("Learning Rate")
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].yaxis.set_major_locator(mticker.MaxNLocator(nbins=8))
    axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1e}"))

    plt.tight_layout()

    file_path = os.path.join(save_dir, f"telemetry_{arch_name}_{regime}_seed{seed}_fold{fold}.png")
    plt.savefig(file_path)
    print(f" [Grafico] Telemetria salva em: {file_path}")
    plt.close()  # Libera RAM
    return file_path

In [ ]:
# =============================================================================
# 3. ESTRATEGIAS DE AUMENTO DE DADOS: CutMix e Puzzle Mix
# =============================================================================
# CutMix: Yun et al., 2019 (https://arxiv.org/abs/1905.04899) - implementacao
# padrao, corta um retangulo aleatorio de uma imagem e cola na outra.
#
# Puzzle Mix: Kim et al., ICML 2020 (https://arxiv.org/abs/2009.06962) -
# IMPORTANTE: esta e' uma reimplementacao SIMPLIFICADA do metodo original.
# Pipeline atual (revisado em 2026-08-13 - ver nota "CORRIGIDO" na celula 1
# e puzzlemix_sanity_check/RELATORIO.md para o diagnostico com imagens):
#   1. saliencia por gradiente de entrada, ||dL/dx||_2 nos 3 canais
#      (input_saliency) - norma L2 de verdade, nao mais soma de |grad|.
#   2. particionamento em blocos (grid PxP) e reducao da saliencia por bloco
#      (_block_reduce).
#   3. decisao BINARIA por bloco (fica com A ou com B) por minimizacao de
#      energia via CORTE MINIMO (_alpha_beta_swap_mask): termo unario =
#      gain = sal_a - sal_b por bloco; termo de suavidade = grade
#      4-conectada (regularizacao "mask-graph" contra mascara fragmentada -
#      a lacuna que a versao anterior deste notebook ja citava na docstring
#      mas nao implementava) MAIS arestas extras nos pares de blocos que o
#      plano de transporte otimo (POT, se disponivel) realmente conectou,
#      ponderadas pela massa transportada. Resolvido com PyMaxflow (busca
#      binaria num vies global 'beta' do termo unario ate' bater a
#      fracao-alvo lambda).
#
#      NOTA sobre o nome "alpha-beta swap": para K=2 rotulos (aqui: bloco
#      vem de A ou de B) o algoritmo geral de troca alpha-beta (Boykov-
#      Veksler-Zabih) se reduz a UM UNICO corte minimo exato - com so' 2
#      rotulos nao ha' o que "trocar" entre pares de rotulos. Por isso um
#      max-flow/min-cut binario (PyMaxflow) resolve exatamente o mesmo
#      problema que pyGCO resolveria aqui, sem a dependencia C++ (pyGCO /
#      gco-wrapper nao tem wheel pre-compilada pra Windows e exige MSVC
#      Build Tools pra compilar do fonte - ver timing.py). Sem PyMaxflow
#      instalado (MAXFLOW_AVAILABLE=False), cai no fallback guloso
#      (_greedy_block_mask) - so' ranking por bloco, sem suavidade.
#
# ACHADO CORRIGIDO: a versao anterior de _ot_block_mask calculava o plano
# de OT corretamente mas decidia keep_a por diag(plano), que e' degenerado
# (transportar um bloco pra ele mesmo custa zero, entao o LP quase sempre
# marca "fica com A" em ~100% dos blocos) - a mascara final saia identica
# ao fallback guloso em 20/20 configuracoes testadas (4 pares x 5 lambdas).
# Aqui o plano de OT so' informa arestas extras do grafo (estrutura
# espacial), nunca decide sozinho o rotulo de um bloco.
#
# Quem precisar de fidelidade total ao paper original (que transporta o
# CONTEUDO dos pixels via o plano de OT, nao so' decide A-ou-B por bloco)
# deve trocar por https://github.com/snu-mllab/PuzzleMix. Aqui o objetivo
# continua sendo isolar o EFEITO DE USAR SALIENCIA (+ suavidade espacial)
# PARA GUIAR A MISTURA, nao replicar o artigo byte a byte - o artigo do
# SIBGRAPI deve declarar essa simplificacao na secao de metodologia.
# =============================================================================

def rand_bbox(size, lam):
    """Amostra uma caixa retangular aleatoria cuja area ~= (1 - lam) da imagem."""
    H, W = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_h, cut_w = int(H * cut_rat), int(W * cut_rat)

    cy, cx = np.random.randint(H), np.random.randint(W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, y2, x1, x2


def pair_indices(labels, rank_lookup, restrict_adjacent):
    """
    Sorteia um parceiro de mistura para cada amostra do batch (permutacao).
    Se restrict_adjacent=True, restringe pares a classes ordinais vizinhas
    (|rank_a - rank_b| <= 1) - ver discussao sobre rotulos intermediarios em
    tarefa ordinal. Amostras sem parceiro valido na permutacao sorteada ficam
    pareadas consigo mesmas (equivale a nao misturar essa amostra).
    """
    n = labels.size(0)
    perm = torch.randperm(n, device=labels.device)
    if not restrict_adjacent:
        return perm

    ranks = torch.as_tensor(rank_lookup, device=labels.device)[labels]
    ranks_perm = ranks[perm]
    ok = (ranks - ranks_perm).abs() <= 1
    perm = torch.where(ok, perm, torch.arange(n, device=labels.device))
    return perm


def cutmix_batch(images, labels, rank_lookup, alpha, restrict_adjacent):
    """
    Retorna (imagens_misturadas, y_a, y_b, lam) com lam = fracao efetiva de
    'images' que sobrou apos o corte (a area colada de y_b e' 1 - lam).
    """
    perm = pair_indices(labels, rank_lookup, restrict_adjacent)
    lam = float(np.random.beta(alpha, alpha))

    mixed = images.clone()
    y1, y2, x1, x2 = rand_bbox(images.size(), lam)
    mixed[:, :, y1:y2, x1:x2] = images[perm][:, :, y1:y2, x1:x2]

    # lam efetivo = area realmente preservada de 'images' (corrige arredondamento)
    box_area = (y2 - y1) * (x2 - x1)
    total_area = images.size(2) * images.size(3)
    lam_eff = 1.0 - box_area / total_area

    return mixed, labels, labels[perm], lam_eff


def input_saliency(model, images, labels, criterion):
    """
    Saliencia por gradiente de entrada: ||dL/dx||_2 nos 3 canais (norma L2
    de verdade - sqrt da soma dos quadrados), normalizada em [0, 1] por
    imagem. Requer um forward+backward EXTRA (fora do passo de otimizacao) -
    e' o principal custo adicional do Puzzle Mix frente a CutMix/baseline.

    ATE' 2026-08-12 isto somava |dL/dx| nos canais (norma L1 por canal, nao
    L2) - corrigido pra bater com a notacao ||.||_2 da metodologia. As duas
    formulas correlacionam ~0.975 pixel a pixel nesta base, mas mudam ~2-6%
    dos blocos escolhidos pela mascara (ver puzzlemix_sanity_check/) - nao
    e' so' uma troca cosmetica.
    """
    was_training = model.training
    model.eval()  # BN/Dropout nao devem acumular estatisticas deste passo extra
    x = images.clone().detach().requires_grad_(True)
    outputs = model(x)
    loss = criterion(outputs, labels)
    grad = torch.autograd.grad(loss, x)[0]
    sal = grad.pow(2).sum(dim=1).sqrt()  # ||grad||_2 nos canais -> (N, H, W)

    flat = sal.flatten(1)
    sal = (sal - flat.min(1)[0].view(-1, 1, 1)) / (
        flat.max(1)[0].view(-1, 1, 1) - flat.min(1)[0].view(-1, 1, 1) + 1e-8
    )
    if was_training:
        model.train()
    return sal.detach()


def _block_reduce(sal, grid):
    """Media da saliencia por bloco -> tensor (N, grid, grid)."""
    N, H, W = sal.shape
    bh, bw = H // grid, W // grid
    sal = sal[:, : bh * grid, : bw * grid]
    sal = sal.view(N, grid, bh, grid, bw).mean(dim=(2, 4))
    return sal  # (N, grid, grid)


def _greedy_block_mask(sal_a, sal_b, lam, grid):
    """
    Fallback sem PyMaxflow: para cada par, ordena os blocos por 'ganho' de
    manter A (sal_a - sal_b) e mantem os top-k blocos de A ate cobrir ~lam
    da area; o resto vem de B. Sem regularizacao espacial (pode fragmentar
    a mascara) - so' entra se MAXFLOW_AVAILABLE=False.
    """
    N = sal_a.size(0)
    n_blocks = grid * grid
    k = int(round(lam * n_blocks))
    gain = (sal_a - sal_b).view(N, -1)
    order = gain.argsort(dim=1, descending=True)
    mask = torch.zeros_like(gain)
    idx = order[:, :k]
    mask.scatter_(1, idx, 1.0)
    return mask.view(N, grid, grid)


def _min_cut_labels(gain, beta, grid, smooth_weight, plan, ot_edge_weight):
    """
    Um corte minimo binario (rotulo A = lado da fonte, rotulo B = lado do
    sumidouro) sobre a grade `grid`x`grid`, com:
      - termo unario: custo de A = relu(beta - gain), custo de B =
        relu(gain - beta) (gain = sal_a - sal_b por bloco; 'beta' e' o vies
        global que controla a fracao de blocos A, ajustado por busca binaria
        em _alpha_beta_swap_mask ate' bater lam);
      - termo de suavidade: grade 4-conectada com peso `smooth_weight`
        (regularizacao "mask-graph" contra mascara fragmentada) MAIS uma
        aresta extra para cada par (i, j) com massa de transporte
        plan[i, j] > 0 (i != j), ponderada por `ot_edge_weight` - os UNICOS
        pares que o LP de transporte otimo considerou "trocaveis" (o plano
        de ot.emd() e' esparso: no maximo n+m-1 entradas nao-nulas).
    Para K=2 rotulos, isto e' exatamente o que o algoritmo alpha-beta swap
    (Boykov-Veksler-Zabih) calcularia - com so' 2 rotulos nao ha' "troca"
    entre pares de rotulos, um unico corte ja' e' a solucao exata.
    """
    n_blocks = grid * grid
    g = maxflow.Graph[float]()
    nodeids = g.add_grid_nodes((grid, grid))
    g.add_grid_edges(nodeids, weights=smooth_weight)
    if plan is not None:
        flat_ids = nodeids.flatten()
        rows, cols = np.nonzero(plan)
        for r, c in zip(rows.tolist(), cols.tolist()):
            if r == c:
                continue
            w = ot_edge_weight * plan[r, c] * n_blocks
            if w > 1e-6:
                g.add_edge(int(flat_ids[r]), int(flat_ids[c]), w, w)
    cost_a = np.maximum(0.0, beta - gain).reshape(grid, grid)
    cost_b = np.maximum(0.0, gain - beta).reshape(grid, grid)
    g.add_grid_tedges(nodeids, cost_b, cost_a)  # cap.fonte=custo de B, cap.sumidouro=custo de A
    g.maxflow()
    is_sink_side = g.get_grid_segments(nodeids)  # True = lado do sumidouro = rotulo B
    return (~is_sink_side).astype(np.float64).flatten()  # 1.0 = A


def _alpha_beta_swap_mask(sal_a, sal_b, lam, grid, use_ot_edges,
                          smooth_frac=0.35, ot_edge_frac=0.7, beta_iters=18):
    """
    Decide, por bloco, se o conteudo final vem de A ou de B, minimizando
    energia (unario por saliencia + suavidade espacial) via corte minimo -
    ver _min_cut_labels. Os pesos de suavidade sao escalados pelo
    desvio-padrao do proprio 'gain' de cada amostra (nao um valor absoluto
    fixo), porque a magnitude do gain depende da formula de saliencia e da
    normalizacao - um peso fixo colapsa a mascara inteira num unico rotulo
    (testado e descartado; ver puzzlemix_sanity_check/).

    Substitui _ot_block_mask (removida em 2026-08-13): aquela funcao
    calculava o plano de OT mas decidia keep_a por diag(plano), o que e'
    degenerado (ver nota da secao 3) - a mascara final saia identica ao
    fallback guloso em todo teste feito. Aqui o plano de OT so' informa
    arestas extras do grafo, nunca decide sozinho o rotulo de um bloco.
    """
    N, g, _ = sal_a.shape
    n_blocks = g * g
    coords = np.array([[r, c] for r in range(g) for c in range(g)], dtype=np.float64)
    cost_mat = ot.dist(coords, coords, metric="euclidean") if use_ot_edges else None

    masks = torch.zeros(N, n_blocks, device=sal_a.device)
    for i in range(N):
        a_np = sal_a[i].cpu().numpy().astype(np.float64)
        b_np = sal_b[i].cpu().numpy().astype(np.float64)
        gain = (a_np - b_np).flatten()
        std = gain.std() + 1e-8
        smooth_weight = smooth_frac * std
        ot_edge_weight = ot_edge_frac * std

        plan = None
        if use_ot_edges:
            a_dist = a_np.flatten() / (a_np.sum() + 1e-8)
            b_dist = b_np.flatten() / (b_np.sum() + 1e-8)
            plan = ot.emd(a_dist, b_dist, cost_mat)

        k_target = int(round(lam * n_blocks))
        lo, hi = -1.0, 1.0
        best_mask, best_diff = None, None
        for _ in range(beta_iters):
            beta = (lo + hi) / 2
            mask = _min_cut_labels(gain, beta, g, smooth_weight, plan, ot_edge_weight)
            k = int(mask.sum())
            diff = abs(k - k_target)
            if best_diff is None or diff < best_diff:
                best_mask, best_diff = mask, diff
            if diff == 0:
                break
            elif k > k_target:   # A demais -> sobe o vies (encarece A)
                lo = beta
            else:                 # A de menos -> desce o vies (barateia A)
                hi = beta
        masks[i] = torch.from_numpy(best_mask.astype(np.float32))
    return masks.view(N, g, g)


def puzzlemix_batch(model, images, labels, rank_lookup, criterion, alpha,
                    grid, restrict_adjacent, use_ot,
                    smooth_frac=None, ot_edge_frac=None):
    """
    Puzzle Mix simplificado (ver docstring da secao). Retorna
    (imagens_misturadas, y_a, y_b, lam_efetivo_por_amostra).
    """
    perm = pair_indices(labels, rank_lookup, restrict_adjacent)
    lam = float(np.random.beta(alpha, alpha))

    sal = input_saliency(model, images, labels, criterion)
    sal_b = sal[perm]
    grid = min(grid, images.size(2), images.size(3))
    sal_blocks_a = _block_reduce(sal, grid)
    sal_blocks_b = _block_reduce(sal_b, grid)

    if MAXFLOW_AVAILABLE:
        mask_blocks = _alpha_beta_swap_mask(
            sal_blocks_a, sal_blocks_b, lam, grid,
            use_ot_edges=(use_ot and POT_AVAILABLE),
            smooth_frac=PUZZLEMIX_SMOOTH_FRAC if smooth_frac is None else smooth_frac,
            ot_edge_frac=PUZZLEMIX_OT_EDGE_FRAC if ot_edge_frac is None else ot_edge_frac,
        )
    else:
        mask_blocks = _greedy_block_mask(sal_blocks_a, sal_blocks_b, lam, grid)

    N, _, H, W = images.shape
    mask = F.interpolate(mask_blocks.unsqueeze(1), size=(H, W), mode="nearest")

    mixed = images * mask + images[perm] * (1 - mask)
    lam_eff = mask.flatten(1).mean(dim=1)  # fracao real de A por amostra, (N,)

    return mixed, labels, labels[perm], lam_eff


def apply_training_regime(regime, model, images, labels, rank_lookup, criterion,
                          restrict_adjacent):
    """
    Dispatcher chamado DENTRO do loop de treino, uma vez por batch.
    Retorna (imagens_para_forward, y_a, y_b, lam) onde lam pode ser escalar
    (CutMix/baseline) ou um tensor (N,) (Puzzle Mix, lam por amostra).
    Para 'baseline', y_a == y_b e lam == 1.0 (sem mistura; loss cai no CE normal).
    """
    if regime == "baseline":
        return images, labels, labels, 1.0

    if np.random.rand() > MIX_PROB:
        return images, labels, labels, 1.0  # este batch nao sera misturado

    if regime == "cutmix":
        return cutmix_batch(images, labels, rank_lookup, MIXUP_ALPHA, restrict_adjacent)

    if regime == "puzzlemix":
        return puzzlemix_batch(model, images, labels, rank_lookup, criterion,
                               MIXUP_ALPHA, PUZZLEMIX_GRID, restrict_adjacent,
                               PUZZLEMIX_USE_OT)

    raise ValueError(f"Regime de treino desconhecido: {regime}")


def mixed_criterion(criterion, outputs, y_a, y_b, lam):
    """
    Loss para batch (possivelmente) misturado. lam pode ser float (mesmo
    lambda para o batch todo, caso CutMix/baseline) ou tensor (N,) com um
    lambda por amostra (caso Puzzle Mix, ja que a fracao de area de cada
    par pode variar levemente por causa do arredondamento do grid).
    """
    if isinstance(lam, float):
        return lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)

    # lam por amostra: usa reducao 'none' e faz a media manualmente
    ce = nn.CrossEntropyLoss(weight=criterion.weight, reduction="none")
    per_sample = lam * ce(outputs, y_a) + (1 - lam) * ce(outputs, y_b)
    return per_sample.mean()

In [ ]:
def run_config(arch_name: str, regime: str, seed: int):
    print("\n" + "#" * 70)
    print(f"# ARQUITETURA={arch_name}  |  REGIME={regime}  |  SEED={seed}")
    print("#" * 70)

    seed_everything(seed)
    pin_memory = (device.type == "cuda")

    # --- Resolucao de LR (NOVO): por (arquitetura, regime); fallback com aviso ---
    lr = LR_BY_ARCH_REGIME.get((arch_name, regime))
    if lr is None:
        lr = LR_DEFAULT_BY_ARCH[arch_name]
        warnings.warn(
            f"LR_BY_ARCH_REGIME nao tem entrada para ({arch_name}, {regime}); "
            f"usando o default de {arch_name} (lr={lr:.1e}). Isso e' um "
            f"PLACEHOLDER - rode o HPO curto (fase 0b) e preencha "
            f"LR_BY_ARCH_REGIME antes de gerar os resultados definitivos."
        )

    # --- Resolucao de batch fisico / acumulacao (NOVO) ---
    batch_size = BATCH_SIZE_BY_ARCH[arch_name]
    accum_steps = ACCUM_STEPS_BY_ARCH[arch_name]
    assert batch_size * accum_steps == EFFECTIVE_BATCH_SIZE, (
        f"BATCH_SIZE_BY_ARCH x ACCUM_STEPS_BY_ARCH para '{arch_name}' "
        f"({batch_size} x {accum_steps}) != EFFECTIVE_BATCH_SIZE ({EFFECTIVE_BATCH_SIZE}); "
        f"corrija a config para manter o batch efetivo comparavel entre arquiteturas."
    )
    print(f"Checkpoint metric: {CHECKPOINT_METRIC} | LR={lr:.1e} | "
          f"batch fisico={batch_size} x accum={accum_steps} (efetivo={EFFECTIVE_BATCH_SIZE}) | "
          f"AMP={USE_AMP} | mistura restrita a classes vizinhas={RESTRICT_ADJACENT_MIX}")

    data_transforms = {
        "treino": transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(40),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
        "teste": transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
    }
    # CutMix/Puzzle Mix entram DEPOIS deste pipeline (sobre o batch ja
    # transformado), dentro do loop de treino - nunca em validacao/teste.

    if not os.path.isdir(BASE_DIR):
        raise FileNotFoundError(f"BASE_DIR nao encontrado: {os.path.abspath(BASE_DIR)}")

    full_aug   = datasets.ImageFolder(root=BASE_DIR, transform=data_transforms["treino"])
    full_plain = datasets.ImageFolder(root=BASE_DIR, transform=data_transforms["teste"])
    num_classes = len(full_aug.classes)
    class_names = full_aug.classes
    rank_lookup = ordinal_ranks(class_names)
    all_labels  = np.array(full_aug.targets)

    classes_idx, classes_count = np.unique(all_labels, return_counts=True)
    print(f"\n--- Dataset: {len(all_labels)} imagens totais, {num_classes} classes ---")
    for c, cnt in zip(classes_idx, classes_count):
        print(f"    {class_names[c]:<12}: {cnt} imagens")

    # Holdout estratificado: random_state=seed -> IDENTICO entre arquiteturas
    # e regimes quando a seed e' a mesma (mesma logica pareada do notebook
    # original, so' que agora "pareia" arquitetura x regime em vez de scheduler).
    cv_idx, test_idx = train_test_split(
        np.arange(len(all_labels)), test_size=TEST_SIZE,
        stratify=all_labels, random_state=seed,
    )
    train_labels = all_labels[cv_idx]
    test_data = Subset(full_plain, test_idx)
    test_loader = DataLoader(test_data, batch_size=batch_size, num_workers=NUM_WORKERS,
                             pin_memory=pin_memory, shuffle=False)

    print(f"--- Split holdout (test_size={TEST_SIZE}): "
          f"treino/CV={len(cv_idx)} imagens | teste={len(test_idx)} imagens ---")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    cv_metrics = {"f1": [], "auc": [], "score": []}
    ckpt_paths = []

    gradcam_layer = get_gradcam_layer(arch_name)
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))

    for fold, (train_pos, val_pos) in enumerate(
            skf.split(np.zeros(len(train_labels)), train_labels)):
        print(f"\n === Fold {fold + 1}/{N_FOLDS} ({arch_name}/{regime}, seed {seed}) ===")
        train_idx = cv_idx[train_pos]
        val_idx   = cv_idx[val_pos]
        print(f"     treino={len(train_idx)} imagens | validacao={len(val_idx)} imagens")
        train_subset = Subset(full_aug,   train_idx)
        val_subset   = Subset(full_plain, val_idx)
        fold_seed = seed * 1000 + fold + 1
        seed_everything(fold_seed)
        fold_gen = make_loader_generator(fold_seed)

        train_loader = DataLoader(train_subset, batch_size=batch_size, num_workers=NUM_WORKERS,
                                  pin_memory=pin_memory, drop_last=True, shuffle=True,
                                  worker_init_fn=seed_worker, generator=fold_gen)
        val_loader   = DataLoader(val_subset, batch_size=batch_size, num_workers=NUM_WORKERS,
                                  pin_memory=pin_memory, shuffle=False)

        model = build_model(arch_name, num_classes).to(device)
        decomposable_layers = get_decomposable_layers(model, arch_name)

        fold_train_labels = train_labels[train_pos]
        cw = compute_class_weight("balanced",
                                  classes=np.unique(fold_train_labels),
                                  y=fold_train_labels)
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(cw, dtype=torch.float32, device=device))
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        scheduler = make_scheduler(optimizer)  # Cosine, fixo, passo por epoca

        best_score, best_val_auc, best_val_f1 = -np.inf, 0.0, 0.0
        best_val_clin = None
        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lrs": []}
        ckpt_path = os.path.join(
            OUTPUT_DIR, f"{arch_name}_{regime}_seed{seed}_fold{fold + 1}_best.pth")

        for epoch in range(EPOCHS):
            # --- treino ---
            model.train()
            train_loss, train_correct, train_total = 0.0, 0, 0
            optimizer.zero_grad(set_to_none=True)
            for step, (images, labels) in enumerate(train_loader):
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                # --- mistura (NOVO): baseline/cutmix/puzzlemix, so' no treino ---
                mixed_images, y_a, y_b, lam = apply_training_regime(
                    regime, model, images, labels, rank_lookup, criterion,
                    RESTRICT_ADJACENT_MIX
                )

                with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                    outputs = model(mixed_images)
                    loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
                    loss_to_backward = loss / accum_steps

                scaler.scale(loss_to_backward).backward()
                if (step + 1) % accum_steps == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

                batch_size_actual = labels.size(0)
                train_loss += loss.item() * batch_size_actual
                # Acuracia de treino contra y_a (rotulo primario do par) - so'
                # sinal diagnostico; batches nao misturados tem y_a == labels.
                train_correct += (torch.argmax(outputs, 1) == y_a).sum().item()
                train_total += batch_size_actual
            # garante que nenhum gradiente acumulado fique pendente no fim da
            # epoca (caso o numero de batches nao seja multiplo de accum_steps)
            if (step + 1) % accum_steps != 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            avg_train_loss = train_loss / train_total
            train_acc = train_correct / train_total

            # --- validacao (NUNCA misturada) ---
            model.eval()
            val_loss, val_total = 0.0, 0
            v_labels, v_preds, v_probs = [], [], []
            with torch.no_grad():
                for images, labels in val_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    outputs = model(images)
                    batch_size_actual = labels.size(0)
                    val_loss += criterion(outputs, labels).item() * batch_size_actual
                    val_total += batch_size_actual
                    v_labels.extend(labels.cpu().numpy())
                    v_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
                    v_probs.extend(F.softmax(outputs, 1).cpu().numpy())

            avg_val_loss = val_loss / val_total
            val_f1 = f1_score(v_labels, v_preds, average="macro")
            val_acc = accuracy_score(v_labels, v_preds)
            try:
                val_auc = roc_auc_score(v_labels, v_probs,
                                        multi_class="ovr", average="macro")
            except ValueError:
                val_auc = float("nan")
            selection_score, val_clin = checkpoint_selection_score(
                val_auc, val_f1, v_labels, v_preds, rank_lookup
            )

            scheduler.step()  # Cosine: sempre por epoca (ver make_scheduler)
            current_lr = optimizer.param_groups[0]['lr']

            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
            history["lrs"].append(current_lr)

            print(f"Ep {epoch + 1:02d}/{EPOCHS} | "
                  f"LR {current_lr:.2e} | "
                  f"TrainLoss {avg_train_loss:.4f} | TrainAcc(y_a) {train_acc:.4f} | "
                  f"ValLoss {avg_val_loss:.4f} | ValAcc {val_acc:.4f} | "
                  f"F1 {val_f1:.4f} | AUC {val_auc:.4f} | "
                  f"QWK {val_clin['qwk']:.4f} | Score {selection_score:.4f}")

            if selection_score > best_score:
                best_score, best_val_auc, best_val_f1 = selection_score, val_auc, val_f1
                best_val_clin = val_clin
                torch.save(model.state_dict(), ckpt_path)

        if not os.path.isfile(ckpt_path):
            warnings.warn(f"Nenhum checkpoint selecionado em {ckpt_path}; salvando estado final do fold.")
            torch.save(model.state_dict(), ckpt_path)
        plot_fold_telemetry(history, fold + 1, arch_name, regime, seed)
        print(f"     melhor checkpoint: score={best_score:.4f} | "
              f"AUC={best_val_auc:.4f} | F1={best_val_f1:.4f}")

        cv_metrics["auc"].append(best_val_auc)
        cv_metrics["f1"].append(best_val_f1)
        cv_metrics["score"].append(best_score)
        ckpt_paths.append(ckpt_path)

    auc_mean, auc_std = np.mean(cv_metrics["auc"]), np.std(cv_metrics["auc"])
    f1_mean,  f1_std  = np.mean(cv_metrics["f1"]),  np.std(cv_metrics["f1"])
    score_mean, score_std = np.mean(cv_metrics["score"]), np.std(cv_metrics["score"])
    print(f"\n--- CV {arch_name}/{regime}/seed{seed}: "
          f"AUC {auc_mean:.4f}±{auc_std:.4f} | "
          f"F1 {f1_mean:.4f}±{f1_std:.4f} | "
          f"Score {score_mean:.4f}±{score_std:.4f} ---")

    # =========================================================================
    # AVALIACAO NO TEST SET (UMA unica vez, listas locais; sem mistura)
    # =========================================================================
    ensemble = []
    for p in ckpt_paths:
        m = build_model(arch_name, num_classes).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True))
        m.eval()
        ensemble.append(m)

    test_labels, test_preds, test_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            batch_probs = torch.zeros((images.size(0), num_classes), device=device)
            for m in ensemble:
                batch_probs += F.softmax(m(images), dim=1)
            batch_probs /= N_FOLDS
            test_labels.extend(labels.numpy())
            test_preds.extend(torch.argmax(batch_probs, 1).cpu().numpy())
            test_probs.extend(batch_probs.cpu().numpy())

    test_f1  = f1_score(test_labels, test_preds, average="macro")
    test_auc = roc_auc_score(test_labels, test_probs, multi_class="ovr", average="macro")
    cm_test  = confusion_matrix(test_labels, test_preds, labels=range(num_classes))
    clin = clinical_metrics(test_labels, test_preds, rank_lookup)

    print(f">>> TEST {arch_name}/{regime}/seed{seed}: F1 {test_f1:.4f} | "
          f"AUC {test_auc:.4f} | QWK {clin['qwk']:.4f} | "
          f"Erros>=2 {clin['danger_count']} ({clin['danger_rate']*100:.1f}%)")

    return {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Architecture": arch_name,
        "Training_Regime": regime,
        "Seed": seed,
        "Folds": N_FOLDS, "Epochs": EPOCHS,
        "Batch_Size_Effective": EFFECTIVE_BATCH_SIZE,
        "Batch_Size_Physical": batch_size, "Accum_Steps": accum_steps,
        "LR": lr, "Weight_Decay": WEIGHT_DECAY, "Optimizer": "AdamW",
        "Scheduler": SCHEDULER_NAME,
        "Mixup_Alpha": MIXUP_ALPHA, "Mix_Prob": MIX_PROB,
        "Restrict_Adjacent_Mix": RESTRICT_ADJACENT_MIX,
        "Checkpoint_Metric": CHECKPOINT_METRIC,
        "Classes": str(class_names), "Test_Size": TEST_SIZE,
        "GradCAM_Layer": gradcam_layer,
        "Decomposable_Conv_Count": len(decomposable_layers),
        "Val_CV_AUC_Mean": f"{auc_mean:.4f}", "Val_CV_AUC_Std": f"{auc_std:.4f}",
        "Val_CV_F1_Mean":  f"{f1_mean:.4f}",  "Val_CV_F1_Std":  f"{f1_std:.4f}",
        "Val_CV_Score_Mean": f"{score_mean:.4f}", "Val_CV_Score_Std": f"{score_std:.4f}",
        "Test_Ensemble_AUC": f"{test_auc:.4f}", "Test_Ensemble_F1": f"{test_f1:.4f}",
        "Test_QWK": f"{clin['qwk']:.4f}", "Test_Accuracy": f"{clin['acc']:.4f}",
        "Test_Danger_Count": clin["danger_count"],
        "Test_Danger_Rate": f"{clin['danger_rate']:.4f}",
        "Test_Underest_Count": clin["underest_count"],
        "Test_Underest_Rate": f"{clin['underest_rate']:.4f}",
        "Test_Confusion_Matrix": str(cm_test.tolist()),
    }

In [ ]:
# =============================================================================
# 5. LOOP DA MATRIZ EXPERIMENTAL  +  LOGGING
# =============================================================================
# MUDANCA vs. original: "Scheduler" saiu do fieldname principal de resume
# (agora e' fixo e vira so' uma coluna informativa); entraram "Architecture"
# e "Training_Regime". Colunas novas: Batch_Size_Physical/Accum_Steps
# (rastreiam a estrategia de VRAM por arquitetura), Mixup_Alpha/Mix_Prob/
# Restrict_Adjacent_Mix (parametros de mistura), GradCAM_Layer e
# Decomposable_Conv_Count (metadados para os notebooks seguintes).
FIELDNAMES = [
    "Timestamp", "Architecture", "Training_Regime", "Seed", "Folds", "Epochs",
    "Batch_Size_Effective", "Batch_Size_Physical", "Accum_Steps",
    "LR", "Weight_Decay", "Optimizer", "Scheduler",
    "Mixup_Alpha", "Mix_Prob", "Restrict_Adjacent_Mix",
    "Checkpoint_Metric", "Classes", "Test_Size",
    "GradCAM_Layer", "Decomposable_Conv_Count",
    "Val_CV_AUC_Mean", "Val_CV_AUC_Std", "Val_CV_F1_Mean", "Val_CV_F1_Std",
    "Val_CV_Score_Mean", "Val_CV_Score_Std",
    "Test_Ensemble_AUC", "Test_Ensemble_F1", "Test_QWK", "Test_Accuracy",
    "Test_Danger_Count", "Test_Danger_Rate", "Test_Underest_Count",
    "Test_Underest_Rate", "Test_Confusion_Matrix",
]


def log_row(row: dict):
    exists = os.path.isfile(CSV_FILE)
    if exists:
        with open(CSV_FILE, newline="", encoding="utf-8") as f:
            header = next(csv.reader(f), [])
        if header != FIELDNAMES:
            raise ValueError(
                f"CSV existente tem colunas incompativeis com o pipeline atual: {CSV_FILE}"
            )
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def get_completed_configs(csv_path: str = CSV_FILE) -> set:
    """
    Le o CSV de resultados (se existir) e retorna o conjunto de
    (Architecture, Training_Regime, Seed) ja concluidos, para que main()
    possa pular essas combinacoes ao retomar uma execucao interrompida.
    """
    if not os.path.isfile(csv_path):
        return set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {(row["Architecture"], row["Training_Regime"], int(row["Seed"])) for row in reader}


# =============================================================================
# 6. RESUMO ESTATISTICO (le o CSV; nao re-treina)
# =============================================================================
import ast


def _expand_cm(cm):
    """Matriz de confusao -> (y_true, y_pred) repetindo cada celula."""
    yt, yp = [], []
    for i, row in enumerate(cm):
        for j, c in enumerate(row):
            yt += [i] * int(c)
            yp += [j] * int(c)
    return np.array(yt), np.array(yp)

In [ ]:
def _ensure_metric_cols(df):
    """
    Garante colunas Test_QWK / Test_Danger_Rate / Test_Accuracy.
    Se faltarem (ex.: CSVs antigos), recalcula a partir de Test_Confusion_Matrix
    usando a ordem ORDINAL clinica (nao a ordem alfabetica do ImageFolder).
    """
    need = {"Test_QWK", "Test_Danger_Rate", "Test_Accuracy"} - set(df.columns)
    if not need:
        return df
    qwks, drates, accs = [], [], []
    for _, r in df.iterrows():
        try:
            class_names = ast.literal_eval(r["Classes"])
        except Exception:
            class_names = ORDINAL_ORDER
        rank_lookup = ordinal_ranks(class_names)
        cm = np.array(ast.literal_eval(r["Test_Confusion_Matrix"]))
        yt, yp = _expand_cm(cm.tolist())
        m = clinical_metrics(yt, yp, rank_lookup)
        qwks.append(m["qwk"]); drates.append(m["danger_rate"]); accs.append(m["acc"])
    if "Test_QWK" not in df.columns:          df["Test_QWK"] = qwks
    if "Test_Danger_Rate" not in df.columns:  df["Test_Danger_Rate"] = drates
    if "Test_Accuracy" not in df.columns:     df["Test_Accuracy"] = accs
    return df


def summarize_experiment_matrix(csv_path: str = CSV_FILE):
    """
    MUDANCA vs. summarize_ablation() original: agrupa por
    (Architecture, Training_Regime) em vez de por Scheduler; os testes
    pareados (Wilcoxon/t/Friedman) agora comparam os REGIMES DE TREINO
    dentro de cada arquitetura (pareados por seed) - e' essa a comparacao
    que sustenta o artigo (baseline vs. cutmix vs. puzzlemix).
    """
    import pandas as pd
    try:
        from scipy import stats
        have_scipy = True
    except ImportError:
        have_scipy = False

    if not os.path.isfile(csv_path):
        print(f"[aviso] CSV nao encontrado: {csv_path}")
        print("        Rode main(run_training=True) para gerar os resultados desta configuracao.")
        return

    df = pd.read_csv(csv_path)
    df = _ensure_metric_cols(df)
    for col in ["Test_QWK", "Test_Ensemble_F1", "Test_Ensemble_AUC",
                "Test_Danger_Rate", "Test_Accuracy"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    metrics = [("Test_QWK", "QWK", "max"),
               ("Test_Ensemble_F1", "F1", "max"),
               ("Test_Ensemble_AUC", "AUC", "max"),
               ("Test_Accuracy", "Acc", "max"),
               ("Test_Danger_Rate", "Erro>=2", "min")]

    for arch in [a for a in ARCHITECTURES if a in df["Architecture"].unique()]:
        sub_arch = df[df["Architecture"] == arch]

        print("\n" + "=" * 78)
        print(f" RESUMO POR REGIME DE TREINO - arquitetura: {arch}  (media +/- desvio entre seeds)")
        print("=" * 78)
        order = [r for r in TRAINING_REGIMES if r in sub_arch["Training_Regime"].unique()]
        head = f"{'Regime':<16}" + "".join(f"{lbl:<16}" for _, lbl, _ in metrics) + "n"
        print(head)
        print("-" * len(head))
        for regime in order:
            g = sub_arch[sub_arch["Training_Regime"] == regime]
            cells = ""
            for col, _, _ in metrics:
                cells += f"{g[col].mean():.4f}+/-{g[col].std(ddof=1):.3f}  "
            print(f"{regime:<16}{cells}{len(g)}")

        print("\nMelhor media por metrica:")
        for col, lbl, direction in metrics:
            means = sub_arch.groupby("Training_Regime")[col].mean()
            best = means.idxmin() if direction == "min" else means.idxmax()
            print(f"  {lbl:<10} -> {best}  ({means[best]:.4f})")

        if not have_scipy:
            print("\n[aviso] scipy nao instalado: pulei os testes de significancia.")
            print("        instale com: pip install scipy")
            continue

        print("\n" + "=" * 78)
        print(f" TESTES DE SIGNIFICANCIA PAREADOS (pareados por seed) - {arch}")
        print("=" * 78)
        n_seeds = sub_arch["Seed"].nunique()
        print(f"n_seeds={n_seeds}.", end=" ")
        if n_seeds < 6:
            print("ATENCAO: com n<6, o Wilcoxon nao alcanca p<0.05 mesmo no caso\n"
                  "         extremo (p minimo bilateral = 0.0625). Trate como exploratorio;\n"
                  "         para conclusao formal, aumente o numero de seeds.")
        else:
            print("")

        for col, lbl, direction in metrics:
            piv = sub_arch.pivot_table(index="Seed", columns="Training_Regime",
                                       values=col, aggfunc="mean")
            regs = [r for r in order if r in piv.columns]
            sub = piv[regs].dropna(axis=0, how="any")
            if len(sub) < 2 or len(regs) < 2:
                print(f"\n[{lbl}] dados insuficientes para teste pareado.")
                continue

            print(f"\n[{lbl}]  (seeds pareadas usadas: {len(sub)})")
            if len(regs) >= 3:
                try:
                    chi2, p = stats.friedmanchisquare(*[sub[r].values for r in regs])
                    print(f"  Friedman (omnibus): chi2={chi2:.3f}  p={p:.4f}")
                except Exception as e:
                    print(f"  Friedman: nao aplicavel ({e})")
            better = max if direction == "max" else min
            for a in range(len(regs)):
                for b in range(a + 1, len(regs)):
                    xa, xb = sub[regs[a]].values, sub[regs[b]].values
                    diff = xa - xb
                    try:
                        w_p = stats.wilcoxon(xa, xb, zero_method="wilcox").pvalue
                    except ValueError:
                        w_p = float("nan")
                    t_p = stats.ttest_rel(xa, xb).pvalue
                    win = regs[a] if better(xa.mean(), xb.mean()) == xa.mean() else regs[b]
                    print(f"    {regs[a]:<14} vs {regs[b]:<14} | "
                          f"d_medio={diff.mean():+.4f} | Wilcoxon p={w_p:.4f} | "
                          f"t-pareado p={t_p:.4f} | tende a favor de: {win}")

    print("\n[nota] QWK/F1/AUC/Acc: maior e' melhor. Erro>=2: menor e' melhor.")
    print("[nota] Comparacoes multiplas: os testes acima sao EXPLORATORIOS. Para")
    print("       a hipotese principal do artigo, pre-registre os thresholds/metricas")
    print("       formais e aplique correcao (ex.: Holm) so' sobre esse subconjunto.")
    print("=" * 78)

In [ ]:
def main(run_training: bool = True):
    if run_training:
        completed = get_completed_configs(CSV_FILE)
        if completed:
            print(f"[resume] {len(completed)} combinacoes ja concluidas em {CSV_FILE}; serao puladas.")

        total_planned = 0
        # Loop: arquitetura (externo) -> regime -> seed (interno). Para uma
        # dada seed, todos os regimes de uma arquitetura veem o MESMO split
        # de teste e a MESMA particao de folds (comparacao pareada).
        for arch_name in ARCHITECTURES:
            seeds = SEEDS_BY_ARCH.get(arch_name, DEFAULT_SEEDS_FOR_NEW_ARCH)
            if arch_name not in SEEDS_BY_ARCH:
                print(f"[aviso] '{arch_name}' nao esta em SEEDS_BY_ARCH; usando "
                      f"DEFAULT_SEEDS_FOR_NEW_ARCH={DEFAULT_SEEDS_FOR_NEW_ARCH}.")
            for regime in TRAINING_REGIMES:
                for seed in seeds:
                    total_planned += 1
                    if (arch_name, regime, seed) in completed:
                        print(f"[SKIP] {arch_name}/{regime}/seed{seed} ja esta em {CSV_FILE}")
                        continue
                    row = run_config(arch_name, regime, seed)
                    log_row(row)
                    completed.add((arch_name, regime, seed))
                    print(f"[LOG] {arch_name}/{regime}/seed{seed} salvo em {CSV_FILE}")
        print("\n" + "=" * 70)
        print(f"Matriz concluida: {total_planned} experimentos planejados.")
        print(f"Resultados em: {CSV_FILE}")
        print("=" * 70)

    # Resumo estatistico ao final (sempre roda; le o CSV gerado)
    summarize_experiment_matrix(CSV_FILE)


if __name__ == "__main__":
    import sys
    # --summary [caminho.csv]  -> so resume um CSV existente (nao re-treina).
    if len(sys.argv) > 1 and sys.argv[1] == "--summary":
        path = sys.argv[2] if len(sys.argv) > 2 else CSV_FILE
        summarize_experiment_matrix(path)
        sys.exit(0)
    main()

In [ ]:
import glob, torch, torch.nn.functional as F
from architectures import build_model

# ache o checkpoint do regime baseline
cands = glob.glob("sibgrapi_ckpt_clinical_composite/**/*baseline*.pt*", recursive=True)
print(cands)                      # confira qual é antes de seguir
CKPT = cands[0]

model = build_model("vgg16_bn", num_classes=4, pretrained=False).to(device)
sd = torch.load(CKPT, map_location=device)
sd = {k.replace("classifier.", "head.fc."): v for k, v in sd.items()}
model.load_state_dict(sd)
model.eval()

In [ ]:
"""
medir_saliencia.py — concentração da saliência no OED, através das seeds.

Cole como célula do pipeline_sibgrapi.ipynb (depois das células que definem
BASE_DIR, TEST_SIZE, N_FOLDS, BATCH_SIZE_BY_ARCH, NUM_WORKERS e device).

Três correções em relação à versão anterior:

1. MEDE NA VALIDAÇÃO, NÃO NO TREINO. O modelo já viu o split de treino: a loss
   ali é baixa, os gradientes são pequenos e a concentração medida fica
   otimista. O split de validação do próprio fold é o mesmo dado, não visto.

2. PISO UNIFORME POR GRADE. Como k = round(0.1·n) é arredondado, o piso é k/n e
   só cai perto de 0,10 em grades finas: 2×2 -> 0,250; 4×4 -> 0,125;
   8×8 -> 0,094; 16×16 -> 0,102. Comparar tudo com "0,10" superestima a
   concentração nas grades grossas.

3. TRANSFORM DETERMINÍSTICO. Rotação, crop e jitter aleatórios só adicionam
   variância a uma medição que se quer estável. (Se quiser caracterizar a
   saliência exatamente como o mixup a vê durante o treino, troque para o
   train_tfm — mas então rode mais batches.)
"""

import glob
import re

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from architectures import build_model

GRIDS = (2, 4, 8, 16)
MAX_BATCHES = 20
FOLD_ALVO = 1          # fixe o fold, varie a seed
SPLIT = "val"          # "val" (recomendado) ou "train" (contaminado por memorização)

eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
full = datasets.ImageFolder(root=BASE_DIR, transform=eval_tfm)          # noqa: F821
all_labels = np.array(full.targets)


def split_indices(seed: int, fold: int):
    """Reproduz o split do run_config() para (seed, fold)."""
    cv_idx, _test_idx = train_test_split(
        np.arange(len(all_labels)), test_size=TEST_SIZE,                 # noqa: F821
        stratify=all_labels, random_state=seed)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)  # noqa: F821
    y_cv = all_labels[cv_idx]
    for f, (tr, va) in enumerate(skf.split(np.zeros(len(y_cv)), y_cv), start=1):
        if f == fold:
            return cv_idx[tr], cv_idx[va]
    raise ValueError(f"fold {fold} não encontrado (N_FOLDS={N_FOLDS})")   # noqa: F821


def concentracao(model, loader, grids=GRIDS, max_batches=MAX_BATCHES):
    """
    Massa de saliência nos k=round(0.1·n) blocos mais quentes, por grade.
    Saliência = sqrt(mean(grad², canais)) — a mesma fórmula do Puzzle Mix oficial.
    """
    acc = {g: [] for g in grids}
    for i, (xb, yb) in enumerate(loader):
        if i >= max_batches:
            break
        xb, yb = xb.to(device), yb.to(device)                            # noqa: F821
        xv = xb.clone().requires_grad_(True)
        F.cross_entropy(model(xv), yb).backward()
        sal = torch.sqrt(torch.mean(xv.grad ** 2, dim=1))
        for g in grids:
            p = F.avg_pool2d(sal.unsqueeze(1), sal.shape[-1] // g).flatten(1)
            p = p / p.sum(1, keepdim=True).clamp_min(1e-12)
            k = max(1, round(0.1 * p.shape[1]))
            acc[g].append(p.topk(k, 1).values.sum(1).mean().item())
        model.zero_grad(set_to_none=True)
    return {g: float(np.mean(v)) for g, v in acc.items()}


def piso_uniforme(g: int) -> float:
    n = g * g
    return max(1, round(0.1 * n)) / n


# --------------------------------------------------------------------------

cands = sorted(glob.glob("sibgrapi_ckpt_clinical_composite/**/*baseline*.pt*",
                         recursive=True))
resultados = {}

for ck in cands:
    m = re.search(r"seed(\d+)_fold(\d+)", ck)
    if not m:
        continue
    seed, fold = int(m.group(1)), int(m.group(2))
    if fold != FOLD_ALVO:
        continue

    sd = torch.load(ck, map_location=device, weights_only=True)          # noqa: F821
    sd = sd.get("model", sd.get("state_dict", sd)) if isinstance(sd, dict) else sd
    sd = {k.replace("classifier.", "head.fc."): v for k, v in sd.items()}
    model = build_model("vgg16_bn", num_classes=4, pretrained=False).to(device)  # noqa: F821
    model.load_state_dict(sd)
    model.eval()

    tr_idx, va_idx = split_indices(seed, fold)
    idx = va_idx if SPLIT == "val" else tr_idx
    loader = DataLoader(Subset(full, idx),
                        batch_size=BATCH_SIZE_BY_ARCH["vgg16_bn"],       # noqa: F821
                        shuffle=False, num_workers=NUM_WORKERS)          # noqa: F821

    resultados[seed] = concentracao(model, loader)
    print(f"seed {seed:>4} fold {fold} ({SPLIT}, {len(idx)} imagens): "
          + "  ".join(f"{g}x{g}={resultados[seed][g]:.3f}" for g in GRIDS))

# --------------------------------------------------------------------------

print("\n" + "=" * 68)
print(f"CONCENTRAÇÃO DA SALIÊNCIA — OED, VGG16-BN baseline, split={SPLIT}, "
      f"{len(resultados)} seeds")
print("=" * 68)
print(f"{'grade':<8}{'blocos':>8}{'piso':>9}{'medido':>18}{'razao':>10}")
for g in GRIDS:
    vals = np.array([resultados[s][g] for s in resultados])
    base = piso_uniforme(g)
    print(f"{f'{g}x{g}':<8}{g*g:>8}{base:>9.3f}"
          f"{f'{vals.mean():.3f} +/- {vals.std(ddof=1) if len(vals) > 1 else 0:.3f}':>18}"
          f"{vals.mean() / base:>9.2f}x")

print("\nrazao ~1.0 -> saliencia plana: o unario empata, o prior decide sozinho,")
print("               a mascara colapsa e o Puzzle Mix degenera em input mixup.")
print("razao >>1.0 -> ha estrutura para o unario explorar; se o metodo ainda")
print("               falhar, a causa nao e' a premissa do dominio.")
print("\nATENCAO: sem um dominio focal medido do mesmo jeito, a razao nao tem")
print("escala. Rode o mesmo probe no PCam para calibrar antes de citar no artigo.")

In [ ]:
"""
medir_saliencia.py — concentração da saliência no OED, através das seeds.

Cole como célula do pipeline_sibgrapi.ipynb (depois das células que definem
BASE_DIR, TEST_SIZE, N_FOLDS, BATCH_SIZE_BY_ARCH, NUM_WORKERS e device).

Três correções em relação à versão anterior:

1. MEDE NA VALIDAÇÃO, NÃO NO TREINO. O modelo já viu o split de treino: a loss
   ali é baixa, os gradientes são pequenos e a concentração medida fica
   otimista. O split de validação do próprio fold é o mesmo dado, não visto.

2. PISO UNIFORME POR GRADE. Como k = round(0.1·n) é arredondado, o piso é k/n e
   só cai perto de 0,10 em grades finas: 2×2 -> 0,250; 4×4 -> 0,125;
   8×8 -> 0,094; 16×16 -> 0,102. Comparar tudo com "0,10" superestima a
   concentração nas grades grossas.

3. TRANSFORM DETERMINÍSTICO. Rotação, crop e jitter aleatórios só adicionam
   variância a uma medição que se quer estável. (Se quiser caracterizar a
   saliência exatamente como o mixup a vê durante o treino, troque para o
   train_tfm — mas então rode mais batches.)

4. NULO POR PERMUTAÇÃO. O piso k/n é o valor de um mapa PERFEITAMENTE plano, e
   saliência por gradiente nunca é: só a cauda pesada da marginal já gera
   concentração acima de k/n sem nenhuma estrutura espacial. Simulação com
   mapas i.i.d. (sem estrutura): meio-normal dá 1,10× em 16×16; lognormal
   σ=2 dá 1,98×. Ou seja, uma razão de 2,3× sobre k/n pode ser inteiramente
   ruído de alta frequência.
   O nulo certo embaralha os pixels DENTRO de cada mapa: destrói a estrutura
   espacial e preserva exatamente a marginal. A quantidade que sustenta
   argumento é `razao_estrutural = real / permutado`:
     ~1,0  -> saliência é ruído com cauda pesada; a premissa do Puzzle Mix
              não vale no domínio (unário empata, prior decide, máscara colapsa)
     >>1,0 -> há estrutura espacial genuína para o unário explorar
"""

import glob
import re

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from architectures import build_model

GRIDS = (2, 4, 8, 16)
MAX_BATCHES = 20
FOLD_ALVO = 1          # fixe o fold, varie a seed
SPLIT = "val"          # "val" (recomendado) ou "train" (contaminado por memorização)

eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
full = datasets.ImageFolder(root=BASE_DIR, transform=eval_tfm)          # noqa: F821
all_labels = np.array(full.targets)


def split_indices(seed: int, fold: int):
    """Reproduz o split do run_config() para (seed, fold)."""
    cv_idx, _test_idx = train_test_split(
        np.arange(len(all_labels)), test_size=TEST_SIZE,                 # noqa: F821
        stratify=all_labels, random_state=seed)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)  # noqa: F821
    y_cv = all_labels[cv_idx]
    for f, (tr, va) in enumerate(skf.split(np.zeros(len(y_cv)), y_cv), start=1):
        if f == fold:
            return cv_idx[tr], cv_idx[va]
    raise ValueError(f"fold {fold} não encontrado (N_FOLDS={N_FOLDS})")   # noqa: F821


def _top_mass(sal: torch.Tensor, g: int) -> float:
    """Massa nos k=round(0.1·n) blocos mais quentes da grade g×g."""
    p = F.avg_pool2d(sal.unsqueeze(1), sal.shape[-1] // g).flatten(1)
    p = p / p.sum(1, keepdim=True).clamp_min(1e-12)
    k = max(1, round(0.1 * p.shape[1]))
    return p.topk(k, 1).values.sum(1).mean().item()


def _shuffle_spatial(sal: torch.Tensor) -> torch.Tensor:
    """
    Embaralha os pixels DENTRO de cada mapa, independentemente por amostra.
    Destrói a estrutura espacial, preserva a marginal exatamente — é o nulo
    contra o qual a concentração real tem que ser comparada.
    """
    flat = sal.flatten(1)
    idx = torch.rand_like(flat).argsort(dim=1)
    return flat.gather(1, idx).view_as(sal)


def concentracao(model, loader, grids=GRIDS, max_batches=MAX_BATCHES, n_perm=3):
    """
    Retorna {grade: (real, permutado)}.
    Saliência = sqrt(mean(grad², canais)) — a mesma fórmula do Puzzle Mix oficial.
    """
    real = {g: [] for g in grids}
    perm = {g: [] for g in grids}
    for i, (xb, yb) in enumerate(loader):
        if i >= max_batches:
            break
        xb, yb = xb.to(device), yb.to(device)                            # noqa: F821
        xv = xb.clone().requires_grad_(True)
        F.cross_entropy(model(xv), yb).backward()
        sal = torch.sqrt(torch.mean(xv.grad ** 2, dim=1)).detach()
        for g in grids:
            real[g].append(_top_mass(sal, g))
            perm[g].append(float(np.mean(
                [_top_mass(_shuffle_spatial(sal), g) for _ in range(n_perm)])))
        model.zero_grad(set_to_none=True)
    return {g: (float(np.mean(real[g])), float(np.mean(perm[g]))) for g in grids}


def piso_uniforme(g: int) -> float:
    n = g * g
    return max(1, round(0.1 * n)) / n


# --------------------------------------------------------------------------

cands = sorted(glob.glob("sibgrapi_ckpt_clinical_composite/**/*baseline*.pt*",
                         recursive=True))
resultados = {}

for ck in cands:
    m = re.search(r"seed(\d+)_fold(\d+)", ck)
    if not m:
        continue
    seed, fold = int(m.group(1)), int(m.group(2))
    if fold != FOLD_ALVO:
        continue

    sd = torch.load(ck, map_location=device, weights_only=True)          # noqa: F821
    sd = sd.get("model", sd.get("state_dict", sd)) if isinstance(sd, dict) else sd
    sd = {k.replace("classifier.", "head.fc."): v for k, v in sd.items()}
    model = build_model("vgg16_bn", num_classes=4, pretrained=False).to(device)  # noqa: F821
    model.load_state_dict(sd)
    model.eval()

    tr_idx, va_idx = split_indices(seed, fold)
    idx = va_idx if SPLIT == "val" else tr_idx
    loader = DataLoader(Subset(full, idx),
                        batch_size=BATCH_SIZE_BY_ARCH["vgg16_bn"],       # noqa: F821
                        shuffle=False, num_workers=NUM_WORKERS)          # noqa: F821

    resultados[seed] = concentracao(model, loader)
    print(f"seed {seed:>4} fold {fold} ({SPLIT}, {len(idx)} imagens): "
          + "  ".join(f"{g}x{g}={resultados[seed][g][0]:.3f}/{resultados[seed][g][1]:.3f}"
                      for g in GRIDS) + "   (real/permutado)")

# --------------------------------------------------------------------------

print("\n" + "=" * 68)
print(f"CONCENTRAÇÃO DA SALIÊNCIA — OED, VGG16-BN baseline, split={SPLIT}, "
      f"{len(resultados)} seeds")
print("=" * 68)
print(f"{'grade':<7}{'piso':>7}{'real':>17}{'permutado':>17}"
      f"{'r/piso':>9}{'ESTRUTURAL':>12}")
for g in GRIDS:
    r = np.array([resultados[s][g][0] for s in resultados])
    q = np.array([resultados[s][g][1] for s in resultados])
    sd_r = r.std(ddof=1) if len(r) > 1 else 0.0
    sd_q = q.std(ddof=1) if len(q) > 1 else 0.0
    base = piso_uniforme(g)
    print(f"{f'{g}x{g}':<7}{base:>7.3f}"
          f"{f'{r.mean():.3f}+/-{sd_r:.3f}':>17}{f'{q.mean():.3f}+/-{sd_q:.3f}':>17}"
          f"{r.mean() / base:>8.2f}x{r.mean() / q.mean():>11.2f}x")

print("\nA coluna que importa e' ESTRUTURAL (real/permutado), nao r/piso.")
print("  ~1.0  -> saliencia e' ruido com cauda pesada, sem estrutura espacial:")
print("           o unario empata, o prior decide sozinho, a mascara COLAPSA")
print("           e o Puzzle Mix degenera em input mixup. Premissa nao vale.")
print("  >>1.0 -> ha estrutura espacial genuina para o unario explorar; se o")
print("           metodo ainda falhar, a causa nao e' a premissa do dominio.")
print("\nATENCAO: sem um dominio focal medido do mesmo jeito, nem a razao")
print("estrutural tem escala absoluta. Rode o mesmo probe no PCam para calibrar.")

In [ ]:
"""
ablacao_block_nums.py — testa a hipótese de descasamento de escala.

HIPÓTESE: a saliência do OED tem estrutura só em escala fina (razão estrutural
medida: 1,23× em 2×2, 1,63× em 4×4, 1,98× em 8×8, 2,05× em 16×16). O Puzzle Mix
oficial sorteia block_num uniformemente em {2,4,8,16}, então metade dos batches
opera numa grade grossa demais para o unário enxergar o sinal — e ali a máscara
é decidida pelo prior, não pela saliência.

PREDIÇÃO, se a hipótese vale:   coarse  <  full  <  fine

O BRAÇO GROSSO É OBRIGATÓRIO. Sem ele, "fine melhor que full" tem explicação
alternativa banal — menos variabilidade de grade, treino mais estável — que
nada tem a ver com escala. Só o ordenamento completo, com coarse ABAIXO de
full, distingue as duas leituras. Se coarse ≈ fine, a hipótese está errada e
você economiza o segundo dataset.

O braço de controle aleatório acompanha cada grade: se `random_fine` também
melhorar sobre `random_coarse`, o ganho vem da granularidade da máscara e não
da saliência, e a hipótese de escala continua sem suporte.
"""

from __future__ import annotations

import warnings
from typing import Dict, Sequence

from puzzlemix_official import PuzzleMixOfficial
from regimes import blockmix_random, make_regime

# grades por braço. 'full' = default do paper.
ARMS: Dict[str, Sequence[int]] = {
    "full":   (2, 4, 8, 16),
    "fine":   (8, 16),
    "coarse": (2, 4),
}


def build_arms(repo_path: str, mean, std, device: str, num_classes: int,
               arms: Dict[str, Sequence[int]] = None,
               with_random_control: bool = True, **pm_kwargs):
    """
    Devolve (regimes, mixers):
      regimes[nome] = fn(model, x, y) -> (x_mix, y_soft)
      mixers[nome]  = a instância PuzzleMixOfficial, para ler .stats depois

    Todos os braços compartilham hiperparâmetros; só block_nums muda — é a
    única variável do experimento.

    CORRIGIDO (2026-08-13): a construção de PuzzleMixOfficial (que carrega
    o mixup_graph oficial -> `import gco`) agora é protegida por
    try/except. Antes, se `gco` (gco-wrapper) não estivesse instalado -
    caso comum no Windows, onde ele não tem wheel pré-compilada e exige
    MSVC Build Tools para compilar do fonte -, a PRIMEIRA iteração do loop
    já derrubava build_arms() inteiro (ImportError não tratado), e nem os
    braços `random_*` (que não dependem de gco) chegavam a ser criados.
    Agora, um braço puzzlemix_* que falhar ao carregar é OMITIDO (com
    aviso) e os demais - incluindo random_{nome} da mesma grade - seguem
    normalmente.
    """
    arms = arms or ARMS
    regimes, mixers = {}, {}

    for nome, bns in arms.items():
        try:
            mixer = PuzzleMixOfficial(repo_path, mean, std, device,
                                      num_classes=num_classes, block_nums=bns,
                                      **pm_kwargs)
            mixers[f"puzzlemix_{nome}"] = mixer
            regimes[f"puzzlemix_{nome}"] = make_regime("puzzlemix", num_classes,
                                                       puzzle_mixer=mixer)
        except ImportError as e:
            warnings.warn(
                f"puzzlemix_{nome}: nao consegui carregar o PuzzleMix oficial "
                f"({e}); esse braco fica de fora (random_{nome} continua). "
                f"Precisa de gco-wrapper instalado + repo_path apontando "
                f"para um clone valido de snu-mllab/PuzzleMix."
            )
        if with_random_control:
            regimes[f"random_{nome}"] = (
                lambda model, x, y, _b=bns:
                blockmix_random(x, y, num_classes, alpha=1.0, block_nums=_b))

    return regimes, mixers


def collapse_report(mixers: Dict[str, PuzzleMixOfficial]) -> str:
    """
    Telemetria por braço, colhida durante o treino. É a evidência MECANÍSTICA:
    se `coarse` tiver taxa de colapso alta e `fine` baixa, a explicação de
    escala fica demonstrada no nível da máscara, não só inferida do resultado
    final. Chame reset_stats() no início de cada época e leia no fim.
    """
    L = [f"{'braco':<22}{'colapso':>10}{'sal_top10':>12}{'piso':>9}{'concentr.':>12}"]
    for nome, m in mixers.items():
        s = m.stats
        L.append(f"{nome:<22}{s['collapse_rate']:>10.3f}{s['saliency_top10']:>12.3f}"
                 f"{s['saliency_top10_uniform']:>9.3f}{s['saliency_concentration']:>11.2f}x")
    L.append("\ncolapso alto = mascara constante = Puzzle Mix degenerado em input mixup")
    return "\n".join(L)


# --------------------------------------------------------------- integração
# INTEGRADO (2026-08-13): a proxima celula (Secao 7) implementa exatamente
# isso - run_config_ablacao() (CV+ensemble, alvo soft) e main_ablacao()
# (loop arm x seed, resume via CSV proprio, chama collapse_report() e o
# relatorio pre-registrado de stats_protocol.py no final). O esqueleto
# abaixo descreve a ideia; a implementacao completa esta' na Secao 7.
#
#     from ablacao_block_nums import build_arms, collapse_report
#
#     regimes, mixers = build_arms(REPO, MEAN, STD, device, num_classes=4)
#     # nomes gerados: puzzlemix_full/fine/coarse, random_full/fine/coarse
#     # (puzzlemix_* fica de fora, com aviso, se gco-wrapper nao estiver
#     # instalado - ver CORRIGIDO na docstring de build_arms())
#
#     for nome, mix_fn in regimes.items():
#         for seed in SEEDS:
#             ...
#             x_mix, y_soft = mix_fn(model, xb, yb)
#             loss = soft_target_cross_entropy(model(x_mix), y_soft)
#             ...
#         if nome in mixers:
#             row.update(mixers[nome].stats)     # colapso vai para o CSV
#     print(collapse_report(mixers))
#
# CUSTO: 6 regimes contra os 4 anteriores (+ baseline e cutmix = 8 no total).
# Com 10 seeds isso é 80 treinos por arquitetura. Se apertar, corte os controles
# aleatórios de full — `random_fine` e `random_coarse` bastam para o argumento.
#
# ANÁLISE: a comparação pré-registrada é puzzlemix_fine vs puzzlemix_coarse na
# métrica primária (QWK). Acrescente esse par em Protocol.confirmatory_pairs
# ANTES de rodar, senão ele entra como exploratório.

In [ ]:
# =============================================================================
# 7. ABLACAO block_num (PuzzleMix OFICIAL) - integracao com o loop de treino
# =============================================================================
# Liga build_arms()/collapse_report() (celula anterior, ablacao_block_nums.py)
# ao MESMO tipo de loop de treino/CV/ensemble de run_config() (Secao 4), so'
# que treinando com ALVO SOFT (regimes.py / PuzzleMixOfficial) em vez do
# y_a/y_b/lam do braco "puzzlemix" simplificado da Secao 3.
#
# Roda em paralelo, sem mexer no sibgrapi_logs.csv nem nos regimes antigos:
# CSV proprio (sibgrapi_ablacao_block_nums.csv), schema proprio (tem
# Arm/Block_Nums/Collapse_Rate/Saliency_*).
#
# GCO: os bracos 'puzzlemix_*' (mixup_graph oficial) precisam do pacote
# 'gco' (gco-wrapper, o alpha-beta swap real). Sem ele, sao pulados com
# aviso - os bracos 'random_*'/'baseline'/'cutmix' (sem gco) rodam normal.
# =============================================================================
from typing import Sequence

try:
    import gco  # dependencia do mixup_graph oficial (graph-cut / alpha-beta swap)
    GCO_AVAILABLE = True
except ImportError:
    GCO_AVAILABLE = False
    warnings.warn(
        "Pacote 'gco' (gco-wrapper) nao encontrado; os bracos 'puzzlemix_*' "
        "desta ablacao (mixup_graph OFICIAL, nao a versao simplificada da "
        "Secao 3) serao pulados. Precisa compilar gco-wrapper (exige MSVC "
        "Build Tools no Windows - considere WSL/Linux) ou rodar esta celula "
        "em outro ambiente. Os bracos 'random_*'/'baseline'/'cutmix' (sem "
        "gco) continuam funcionando normalmente."
    )

from architectures import build_model as build_model_v2  # noqa: E402 - so' para esta ablacao
from puzzlemix_official import soft_target_cross_entropy
from regimes import make_regime
from stats_protocol import Protocol, analyze, format_report

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Clone local do repo oficial (git clone https://github.com/snu-mllab/PuzzleMix.git);
# troque se o clone estiver em outro lugar. Note: test_novos.py usa um
# caminho Linux (/home/claude/PuzzleMix) que nao existe neste Windows.
PUZZLEMIX_OFFICIAL_REPO = "./PuzzleMix"

ABLACAO_CSV = os.path.join(OUTPUT_DIR, "sibgrapi_ablacao_block_nums.csv")
ABLACAO_FIELDNAMES = [
    "Timestamp", "Architecture", "Arm", "Block_Nums", "Seed", "Folds", "Epochs",
    "Batch_Size_Effective", "Batch_Size_Physical", "Accum_Steps", "LR",
    "Val_CV_AUC_Mean", "Val_CV_AUC_Std", "Val_CV_F1_Mean", "Val_CV_F1_Std",
    "Val_CV_Score_Mean", "Val_CV_Score_Std",
    "Test_Ensemble_AUC", "Test_Ensemble_F1", "Test_QWK", "Test_Accuracy",
    "Test_Danger_Count", "Test_Danger_Rate", "Test_Underest_Count", "Test_Underest_Rate",
    "Test_Confusion_Matrix",
    "Collapse_Rate", "Saliency_Top10", "Saliency_Top10_Uniform", "Saliency_Concentration",
]


def log_ablacao_row(row: dict, csv_path: str = None):
    csv_path = csv_path or ABLACAO_CSV
    exists = os.path.isfile(csv_path)
    if exists:
        with open(csv_path, newline="", encoding="utf-8") as f:
            header = next(csv.reader(f), [])
        if header != ABLACAO_FIELDNAMES:
            raise ValueError(f"CSV existente tem colunas incompativeis: {csv_path}")
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=ABLACAO_FIELDNAMES)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def get_completed_ablacao(csv_path: str = None) -> set:
    csv_path = csv_path or ABLACAO_CSV
    if not os.path.isfile(csv_path):
        return set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {(row["Architecture"], row["Arm"], int(row["Seed"])) for row in reader}


def run_config_ablacao(arch_name: str, arm_name: str, seed: int, mix_fn,
                       mixer=None, block_nums=None, pretrained: bool = True,
                       epochs: int = None, n_folds: int = None,
                       output_dir: str = None):
    """
    Mesma estrutura de run_config() (holdout estratificado + StratifiedKFold
    + ensemble no teste), mas treinando com ALVO SOFT (mix_fn devolve
    (x_misturado, y_soft), regimes.py/PuzzleMixOfficial) em vez de
    y_a/y_b/lam. epochs/n_folds/output_dir tem override so' para smoke test
    rapido (default = EPOCHS/N_FOLDS/OUTPUT_DIR globais, o treino "de verdade").

    Usa architectures.build_model() (build_model_v2), NAO o build_model()
    local da Secao 2 - e' o que puzzlemix_official.py/regimes.py foram
    desenhados para usar (suporta densenet121/efficientnet_b0 tambem, e
    expoe .features do mesmo jeito que a VGGGap para Grad-CAM/decomposicao).
    """
    epochs = EPOCHS if epochs is None else epochs
    n_folds = N_FOLDS if n_folds is None else n_folds
    output_dir = OUTPUT_DIR if output_dir is None else output_dir
    os.makedirs(output_dir, exist_ok=True)

    print("\n" + "#" * 70)
    print(f"# [ABLACAO] ARQUITETURA={arch_name}  |  BRACO={arm_name}  |  SEED={seed}")
    print("#" * 70)

    seed_everything(seed)
    pin_memory = (device.type == "cuda")

    lr = LR_BY_ARCH_REGIME.get((arch_name, arm_name), LR_DEFAULT_BY_ARCH[arch_name])
    batch_size = BATCH_SIZE_BY_ARCH[arch_name]
    accum_steps = ACCUM_STEPS_BY_ARCH[arch_name]

    data_transforms = {
        "treino": transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(40),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]),
        "teste": transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]),
    }
    if not os.path.isdir(BASE_DIR):
        raise FileNotFoundError(f"BASE_DIR nao encontrado: {os.path.abspath(BASE_DIR)}")

    full_aug = datasets.ImageFolder(root=BASE_DIR, transform=data_transforms["treino"])
    full_plain = datasets.ImageFolder(root=BASE_DIR, transform=data_transforms["teste"])
    num_classes = len(full_aug.classes)
    class_names = full_aug.classes
    rank_lookup = ordinal_ranks(class_names)
    all_labels = np.array(full_aug.targets)

    cv_idx, test_idx = train_test_split(
        np.arange(len(all_labels)), test_size=TEST_SIZE,
        stratify=all_labels, random_state=seed,
    )
    train_labels = all_labels[cv_idx]
    test_data = Subset(full_plain, test_idx)
    test_loader = DataLoader(test_data, batch_size=batch_size, num_workers=NUM_WORKERS,
                             pin_memory=pin_memory, shuffle=False)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    cv_metrics = {"f1": [], "auc": [], "score": []}
    ckpt_paths = []
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))

    if mixer is not None:
        mixer.reset_stats()

    for fold, (train_pos, val_pos) in enumerate(
            skf.split(np.zeros(len(train_labels)), train_labels)):
        print(f"\n === [ABLACAO] Fold {fold + 1}/{n_folds} ({arch_name}/{arm_name}, seed {seed}) ===")
        train_idx = cv_idx[train_pos]
        val_idx = cv_idx[val_pos]
        train_subset = Subset(full_aug, train_idx)
        val_subset = Subset(full_plain, val_idx)
        fold_seed = seed * 1000 + fold + 1
        seed_everything(fold_seed)
        fold_gen = make_loader_generator(fold_seed)

        train_loader = DataLoader(train_subset, batch_size=batch_size, num_workers=NUM_WORKERS,
                                  pin_memory=pin_memory, drop_last=True, shuffle=True,
                                  worker_init_fn=seed_worker, generator=fold_gen)
        val_loader = DataLoader(val_subset, batch_size=batch_size, num_workers=NUM_WORKERS,
                                pin_memory=pin_memory, shuffle=False)

        model = build_model_v2(arch_name, num_classes, pretrained=pretrained).to(device)

        fold_train_labels = train_labels[train_pos]
        cw = compute_class_weight("balanced", classes=np.unique(fold_train_labels),
                                  y=fold_train_labels)
        val_criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(cw, dtype=torch.float32, device=device))
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-8)

        best_score, best_val_auc, best_val_f1 = -np.inf, 0.0, 0.0
        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lrs": []}
        ckpt_path = os.path.join(output_dir, f"{arch_name}_{arm_name}_seed{seed}_fold{fold + 1}_best.pth")

        for epoch in range(epochs):
            model.train()
            train_loss, train_correct, train_total = 0.0, 0, 0
            optimizer.zero_grad(set_to_none=True)
            for step, (images, labels) in enumerate(train_loader):
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                # --- mistura OFICIAL (alvo soft), so' no treino ---
                x_mix, y_soft = mix_fn(model, images, labels)

                with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                    outputs = model(x_mix)
                    loss = soft_target_cross_entropy(outputs, y_soft)
                    loss_to_backward = loss / accum_steps

                scaler.scale(loss_to_backward).backward()
                if (step + 1) % accum_steps == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

                bsz = labels.size(0)
                train_loss += loss.item() * bsz
                # acuracia de treino contra o rotulo dominante do alvo soft
                train_correct += (torch.argmax(outputs, 1) == torch.argmax(y_soft, 1)).sum().item()
                train_total += bsz
            if (step + 1) % accum_steps != 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            avg_train_loss = train_loss / train_total
            train_acc = train_correct / train_total

            # --- validacao (NUNCA misturada) ---
            model.eval()
            val_loss, val_total = 0.0, 0
            v_labels, v_preds, v_probs = [], [], []
            with torch.no_grad():
                for images, labels in val_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    outputs = model(images)
                    bsz = labels.size(0)
                    val_loss += val_criterion(outputs, labels).item() * bsz
                    val_total += bsz
                    v_labels.extend(labels.cpu().numpy())
                    v_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
                    v_probs.extend(F.softmax(outputs, 1).cpu().numpy())

            avg_val_loss = val_loss / val_total
            val_f1 = f1_score(v_labels, v_preds, average="macro")
            val_acc = accuracy_score(v_labels, v_preds)
            try:
                val_auc = roc_auc_score(v_labels, v_probs, multi_class="ovr", average="macro")
            except ValueError:
                val_auc = float("nan")
            selection_score, val_clin = checkpoint_selection_score(
                val_auc, val_f1, v_labels, v_preds, rank_lookup)

            scheduler.step()
            current_lr = optimizer.param_groups[0]["lr"]
            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
            history["lrs"].append(current_lr)

            print(f"[ABLACAO] Ep {epoch + 1:02d}/{epochs} | LR {current_lr:.2e} | "
                  f"TrainLoss {avg_train_loss:.4f} | TrainAcc {train_acc:.4f} | "
                  f"ValLoss {avg_val_loss:.4f} | ValAcc {val_acc:.4f} | "
                  f"F1 {val_f1:.4f} | AUC {val_auc:.4f} | QWK {val_clin['qwk']:.4f} | "
                  f"Score {selection_score:.4f}")

            if selection_score > best_score:
                best_score, best_val_auc, best_val_f1 = selection_score, val_auc, val_f1
                torch.save(model.state_dict(), ckpt_path)

        if not os.path.isfile(ckpt_path):
            warnings.warn(f"Nenhum checkpoint selecionado em {ckpt_path}; salvando estado final do fold.")
            torch.save(model.state_dict(), ckpt_path)
        plot_fold_telemetry(history, fold + 1, arch_name, arm_name, seed, save_dir=output_dir)

        cv_metrics["auc"].append(best_val_auc)
        cv_metrics["f1"].append(best_val_f1)
        cv_metrics["score"].append(best_score)
        ckpt_paths.append(ckpt_path)

    auc_mean, auc_std = np.mean(cv_metrics["auc"]), np.std(cv_metrics["auc"])
    f1_mean, f1_std = np.mean(cv_metrics["f1"]), np.std(cv_metrics["f1"])
    score_mean, score_std = np.mean(cv_metrics["score"]), np.std(cv_metrics["score"])
    print(f"\n--- [ABLACAO] CV {arch_name}/{arm_name}/seed{seed}: "
          f"AUC {auc_mean:.4f}+/-{auc_std:.4f} | F1 {f1_mean:.4f}+/-{f1_std:.4f} | "
          f"Score {score_mean:.4f}+/-{score_std:.4f} ---")

    # =========================================================================
    # AVALIACAO NO TEST SET (ensemble dos N_FOLDS checkpoints, sem mistura)
    # =========================================================================
    ensemble = []
    for p in ckpt_paths:
        m = build_model_v2(arch_name, num_classes, pretrained=False).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True))
        m.eval()
        ensemble.append(m)

    test_labels, test_preds, test_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            batch_probs = torch.zeros((images.size(0), num_classes), device=device)
            for m in ensemble:
                batch_probs += F.softmax(m(images), dim=1)
            batch_probs /= n_folds
            test_labels.extend(labels.numpy())
            test_preds.extend(torch.argmax(batch_probs, 1).cpu().numpy())
            test_probs.extend(batch_probs.cpu().numpy())

    test_f1 = f1_score(test_labels, test_preds, average="macro")
    test_auc = roc_auc_score(test_labels, test_probs, multi_class="ovr", average="macro")
    cm_test = confusion_matrix(test_labels, test_preds, labels=range(num_classes))
    clin = clinical_metrics(test_labels, test_preds, rank_lookup)

    print(f">>> [ABLACAO] TEST {arch_name}/{arm_name}/seed{seed}: F1 {test_f1:.4f} | "
          f"AUC {test_auc:.4f} | QWK {clin['qwk']:.4f} | "
          f"Erros>=2 {clin['danger_count']} ({clin['danger_rate']*100:.1f}%)")

    stats = mixer.stats if mixer is not None else {
        "collapse_rate": float("nan"), "saliency_top10": float("nan"),
        "saliency_top10_uniform": float("nan"), "saliency_concentration": float("nan"),
    }

    return {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Architecture": arch_name, "Arm": arm_name,
        "Block_Nums": str(list(block_nums)) if block_nums else "",
        "Seed": seed, "Folds": n_folds, "Epochs": epochs,
        "Batch_Size_Effective": EFFECTIVE_BATCH_SIZE,
        "Batch_Size_Physical": batch_size, "Accum_Steps": accum_steps, "LR": lr,
        "Val_CV_AUC_Mean": f"{auc_mean:.4f}", "Val_CV_AUC_Std": f"{auc_std:.4f}",
        "Val_CV_F1_Mean": f"{f1_mean:.4f}", "Val_CV_F1_Std": f"{f1_std:.4f}",
        "Val_CV_Score_Mean": f"{score_mean:.4f}", "Val_CV_Score_Std": f"{score_std:.4f}",
        "Test_Ensemble_AUC": f"{test_auc:.4f}", "Test_Ensemble_F1": f"{test_f1:.4f}",
        "Test_QWK": f"{clin['qwk']:.4f}", "Test_Accuracy": f"{clin['acc']:.4f}",
        "Test_Danger_Count": clin["danger_count"], "Test_Danger_Rate": f"{clin['danger_rate']:.4f}",
        "Test_Underest_Count": clin["underest_count"],
        "Test_Underest_Rate": f"{clin['underest_rate']:.4f}",
        "Test_Confusion_Matrix": str(cm_test.tolist()),
        "Collapse_Rate": stats["collapse_rate"], "Saliency_Top10": stats["saliency_top10"],
        "Saliency_Top10_Uniform": stats["saliency_top10_uniform"],
        "Saliency_Concentration": stats["saliency_concentration"],
    }


def main_ablacao(arch_name: str = "vgg16_bn", seeds: Sequence[int] = None,
                 repo_path: str = PUZZLEMIX_OFFICIAL_REPO,
                 epochs: int = None, n_folds: int = None,
                 protocol: Protocol = None, csv_path: str = None,
                 output_dir: str = None, only_arms: Sequence[str] = None):
    """
    Roda os 8 bracos (puzzlemix_full/fine/coarse, random_full/fine/coarse,
    baseline, cutmix) x seeds, grava em ABLACAO_CSV com resume (pula
    combinacoes ja concluidas), e ao final imprime collapse_report() +
    o relatorio estatistico PRE-REGISTRADO (stats_protocol). Bracos
    'puzzlemix_*' sao pulados com aviso se GCO_AVAILABLE=False.

    only_arms: roda so' um subconjunto dos 8 (ex.: only_arms=["puzzlemix_coarse"]
    pra iterar rapido num braco so', sem mexer nos outros). Nomes validos:
    puzzlemix_full/fine/coarse, random_full/fine/coarse, baseline, cutmix
    (ver ARMS). O resto (resume, CSV, collapse_report, relatorio) funciona
    igual - a analise estatistica final so' inclui os bracos rodados nesta
    chamada; pra ver o relatorio com TODOS os bracos ja' acumulados no CSV
    (de chamadas anteriores + esta), releia csv_path com pandas.

    Comparacao confirmatoria default: puzzlemix_fine vs puzzlemix_coarse na
    metrica primaria (QWK) - a hipotese de descasamento de escala (ver
    docstring de ablacao_block_nums.py). Declarada e IMPRESSA antes de
    rodar - nao mude confirmatory_pairs depois de ver os resultados.
    """
    seeds = seeds or SEEDS_BY_ARCH.get(arch_name, SEEDS_PRIMARY)
    csv_path = csv_path or ABLACAO_CSV
    protocol = protocol or Protocol(confirmatory_pairs=[("puzzlemix_fine", "puzzlemix_coarse")])
    print(protocol.describe())  # pre-registro: declarado ANTES de rodar
    print()

    num_classes = len(ORDINAL_ORDER)
    regimes, mixers = build_arms(repo_path, IMAGENET_MEAN, IMAGENET_STD, str(device),
                                 num_classes=num_classes)
    # baseline/cutmix entram tambem (8 bracos no total, ver docstring da
    # celula anterior) - mesmos regimes.py, alvo soft, comparaveis aos 6
    # bracos de build_arms() sem confundir com criterio hard-label antigo.
    regimes["baseline"] = make_regime("baseline", num_classes)
    regimes["cutmix"] = make_regime("cutmix", num_classes, alpha=MIXUP_ALPHA)

    if only_arms is not None:
        unknown = set(only_arms) - set(regimes)
        if unknown:
            raise ValueError(
                f"only_arms tem nome(s) desconhecido(s): {sorted(unknown)}. "
                f"Bracos validos: {sorted(regimes)}"
            )
        regimes = {k: v for k, v in regimes.items() if k in only_arms}
        mixers = {k: v for k, v in mixers.items() if k in only_arms}

    completed = get_completed_ablacao(csv_path)
    if completed:
        print(f"[resume] {len(completed)} combinacoes ja concluidas em {csv_path}; serao puladas.")

    results_by_metric: dict = {}
    for arm_name, mix_fn in regimes.items():
        if arm_name.startswith("puzzlemix") and not GCO_AVAILABLE:
            print(f"[SKIP] {arm_name}: precisa de gco-wrapper (nao instalado neste ambiente).")
            continue
        block_nums = ARMS.get(arm_name.split("_", 1)[-1]) if "_" in arm_name and arm_name.split("_", 1)[0] in ("puzzlemix", "random") else None
        mixer = mixers.get(arm_name)
        for seed in seeds:
            if (arch_name, arm_name, seed) in completed:
                print(f"[SKIP] {arch_name}/{arm_name}/seed{seed} ja esta em {csv_path}")
                continue
            row = run_config_ablacao(arch_name, arm_name, seed, mix_fn, mixer=mixer,
                                     block_nums=block_nums, epochs=epochs, n_folds=n_folds,
                                     output_dir=output_dir)
            log_ablacao_row(row, csv_path)
            completed.add((arch_name, arm_name, seed))
            print(f"[LOG] {arch_name}/{arm_name}/seed{seed} salvo em {csv_path}")
            results_by_metric.setdefault(arm_name, {}).setdefault("qwk", []).append(float(row["Test_QWK"]))
            results_by_metric[arm_name].setdefault("erro_ge2", []).append(float(row["Test_Danger_Rate"]))

    if mixers:
        print("\n" + collapse_report(mixers))

    if results_by_metric:
        out = analyze(results_by_metric, protocol)
        print()
        print(format_report(out))

    return results_by_metric

In [ ]:
# =============================================================================
# 8. DISPARO DA ABLACAO block_num (chama de verdade o que a Secao 7 define)
# =============================================================================
# A Secao 7 (celula anterior) so' DEFINE run_config_ablacao()/main_ablacao() -
# igual run_config() (Secao 4) nao se chama sozinha, precisa do main() da
# Secao 5/6 pra rodar. Esta celula e' esse gatilho, no mesmo padrao.
#
# "Rodar tudo" (Run All) agora executa a ablacao tambem, nao so' as sondas
# de saliencia. Resume: combinacoes ja gravadas em ABLACAO_CSV sao puladas.
if __name__ == "__main__":
    main_ablacao()